# 阿瓦隆：约束求解验证 × Qwen3-8B（Colab A100）

按顺序运行全部单元。流程：
1. 克隆 avalonlogs 真实对局，记录 commit；
2. 审计日志是否符合形式化规则；
3. 冻结 LLM 评测集；
4. Z3 与独立穷举交叉验证，不通过则停止；
5. 依次运行 Qwen3-8B 的五种条件并评分；
6. 真实对局分析（仅用 CPU）；
7. 打包下载。

全部原始输出逐批落盘，中断后重新运行同一单元会从断点续跑。不需要授权 Google Drive。完成并下载 ZIP 后，请断开并删除运行时。

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver==4.16.0.0',
                       'transformers==4.57.6', 'safetensors>=0.4.3'])
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), '请选择支持 BF16 的 CUDA GPU'

In [ ]:
import json, sys, os, hashlib
from pathlib import Path
ROOT = Path('/content/avalon_project')
ROOT.mkdir(exist_ok=True)
ASSETS = json.loads('{"avalon/game.py": "\\"\\"\\"Vocabulary, tables and record format for The Resistance: Avalon.\\n\\nA *record* is the public history of one game (JSON-compatible dict):\\n\\n  {\\"n\\": 7,\\n   \\"config\\": {\\"merlin\\": 1, \\"percival\\": 1, \\"servant\\": 2, \\"morgana\\": 1, \\"minion\\": 2},\\n   \\"quests\\": [\\n      {\\"q\\": 1,\\n       \\"proposals\\": [{\\"leader\\": 0, \\"team\\": [2, 4], \\"approve\\": [0, 2, 3, 4], \\"approved\\": true}],\\n       \\"team\\": [2, 4], \\"fails\\": 0, \\"result\\": \\"success\\"},\\n      ...],\\n   \\"end\\": {\\"assassinated\\": 4, \\"outcome\\": \\"good_win\\"}}      # or null while the game runs\\n\\nPlayers are 0..n-1 in seating (clockwise) order; they are shown to people as 1..n.\\n`approve` lists the players who voted to approve (null if unknown).  `fails` is\\nthe number of Fail cards revealed (null if only the result is known).\\n\\nA *view* is the private information of one player:\\n  {\\"player\\": 3, \\"role\\": \\"percival\\", \\"seen\\": [0, 6], \\"lady\\": [[5, false]]}\\n`seen` is what the setup reveal showed that player (meaning depends on the\\nrole, see RULES.md K-*); `lady` lists Lady-of-the-Lake checks made by that player\\nas [target, target_is_evil].\\n\\n*Reveals* are extra alignment facts used by the information-completion\\nanalysis: [[player, is_evil], ...].\\n\\"\\"\\"\\nimport json\\n\\nROLES = (\'merlin\', \'percival\', \'servant\', \'morgana\', \'mordred\', \'oberon\', \'minion\')\\nGOOD_ROLES = (\'merlin\', \'percival\', \'servant\')\\nEVIL_ROLES = (\'morgana\', \'mordred\', \'oberon\', \'minion\')\\nSPECIAL = (\'merlin\', \'percival\', \'morgana\', \'mordred\', \'oberon\')\\n\\n# Rulebook tables (RULES.md S-COUNT, T-SIZE, Q-TWOFAIL)\\nEVIL_COUNT = {5: 2, 6: 2, 7: 3, 8: 3, 9: 3, 10: 4}\\nTEAM_SIZE = {5: (2, 3, 2, 3, 3), 6: (2, 3, 4, 3, 4), 7: (2, 3, 3, 4, 4),\\n             8: (3, 4, 4, 5, 5), 9: (3, 4, 4, 5, 5), 10: (3, 4, 4, 5, 5)}\\n\\n\\ndef fails_required(n, q):\\n    return 2 if q == 4 and n >= 7 else 1\\n\\n\\nZH_ROLE = {\'merlin\': \'梅林\', \'percival\': \'派西维尔\', \'servant\': \'亚瑟的忠臣\', \'morgana\': \'莫甘娜\',\\n           \'mordred\': \'莫德雷德\', \'oberon\': \'奥伯伦\', \'minion\': \'莫德雷德的爪牙\'}\\n\\nDEFAULT_VARIANT = {\\n    \'fail_count_public\': True,   # revealed Quest cards show how many Fails were played\\n    \'merlin_sees_oberon\': True,  # Oberon raises his thumb in the Merlin step\\n}\\n\\n\\ndef player_label(i):\\n    return f\'{i + 1}号\'\\n\\n\\ndef team_label(team):\\n    return \'、\'.join(player_label(i) for i in sorted(team))\\n\\n\\ndef normalize_role(text):\\n    t = str(text).lower()\\n    for key, role in ((\'merlin\', \'merlin\'), (\'percival\', \'percival\'), (\'morgana\', \'morgana\'),\\n                      (\'mordred\', \'mordred\'), (\'oberon\', \'oberon\'), (\'minion\', \'minion\'),\\n                      (\'assassin\', \'minion\'), (\'loyal\', \'servant\'), (\'servant\', \'servant\'),\\n                      (\'follower\', \'servant\'), (\'resistance\', \'servant\'), (\'spy\', \'minion\')):\\n        if key in t:\\n            return role\\n    raise ValueError(f\'unknown role {text!r}\')\\n\\n\\ndef config_ok(n, config):\\n    \\"\\"\\"Static setup check (S-COUNT, S-UNIQUE); returns list of violated rule ids.\\"\\"\\"\\n    bad = []\\n    if n not in EVIL_COUNT:\\n        bad.append(\'S-SIZE\')\\n        return bad\\n    if sum(config.values()) != n:\\n        bad.append(\'S-TOTAL\')\\n    if sum(config.get(r, 0) for r in EVIL_ROLES) != EVIL_COUNT[n]:\\n        bad.append(\'S-COUNT\')\\n    if any(config.get(r, 0) > 1 for r in SPECIAL):\\n        bad.append(\'S-UNIQUE\')\\n    if any(r not in ROLES for r in config):\\n        bad.append(\'S-ROLE\')\\n    return bad\\n\\n\\ndef dumps(x):\\n    return json.dumps(x, ensure_ascii=False, sort_keys=True)\\n\\n\\n# ---------------------------------------------------------------------------\\n# Chinese rendering (used for test cases and LLM inputs)\\n# ---------------------------------------------------------------------------\\ndef render_config(n, config):\\n    parts = []\\n    for r in ROLES:\\n        c = config.get(r, 0)\\n        if c:\\n            parts.append(ZH_ROLE[r] + (f\'×{c}\' if c > 1 else \'\'))\\n    return f\'{n}人局，角色为：\' + \'、\'.join(parts) + f\'（共{EVIL_COUNT.get(n, \\"?\\")}名坏人）。\'\\n\\n\\ndef render_record(rec, show_votes=False):\\n    lines = [render_config(rec[\'n\'], rec[\'config\'])]\\n    for qd in rec.get(\'quests\', []):\\n        q = qd[\'q\']\\n        if show_votes:\\n            for k, p in enumerate(qd.get(\'proposals\', [])):\\n                s = f\'第{q}轮第{k + 1}次提案：队长{player_label(p[\\"leader\\"])}提名{team_label(p[\\"team\\"])}\'\\n                if p.get(\'approve\') is not None:\\n                    s += f\'，赞成者为{team_label(p[\\"approve\\"]) or \\"无\\"}\'\\n                s += \'，通过。\' if p[\'approved\'] else \'，被否决。\'\\n                lines.append(s)\\n        if qd.get(\'result\') is None:\\n            continue\\n        s = f\'第{q}个任务由{team_label(qd[\\"team\\"])}执行\'\\n        if qd.get(\'fails\') is not None:\\n            s += f\'，翻开{qd[\\"fails\\"]}张失败票\'\\n        s += \'，任务\' + (\'成功。\' if qd[\'result\'] == \'success\' else \'失败。\')\\n        lines.append(s)\\n    end = rec.get(\'end\')\\n    if end and end.get(\'assassinated\') is not None:\\n        lines.append(f\'三个任务成功后，刺客刺杀了{player_label(end[\\"assassinated\\"])}，\'\\n                     + (\'好人获胜。\' if end[\'outcome\'] == \'good_win\' else \'坏人获胜。\'))\\n    return \'\\\\n\'.join(lines)\\n\\n\\ndef render_view(view, config):\\n    if view is None:\\n        return \'你是旁观者，只知道公开信息。\'\\n    p, role = view[\'player\'], view[\'role\']\\n    s = f\'你是{player_label(p)}，身份是{ZH_ROLE[role]}。\'\\n    seen = view.get(\'seen\') or []\\n    if role == \'merlin\':\\n        s += f\'开局时你看到的坏人是：{team_label(seen) or \\"无人\\"}。\'\\n    elif role == \'percival\':\\n        s += f\'开局时你看到{team_label(seen)}\' + (\'（其中一人是梅林，另一人是莫甘娜，你分不清）。\' if len(seen) == 2 else \'是梅林。\')\\n    elif role in EVIL_ROLES and role != \'oberon\':\\n        s += f\'开局时你看到的同伴是：{team_label(seen) or \\"无人\\"}。\'\\n    for t, ev in view.get(\'lady\') or []:\\n        s += f\'你用湖中仙女查验了{player_label(t)}，其阵营为{\\"坏人\\" if ev else \\"好人\\"}。\'\\n    return s\\n", "avalon/oracle.py": "\\"\\"\\"Independent executable reading of the Avalon rules (no solver).\\n\\nEnumerates every assignment of the configured role multiset to the seats and\\nkeeps those consistent with the record, the private view and the reveals.\\nIt shares no rule code with smt.py: the tables are typed again here, the\\nquest rule is checked as an inequality (#Fail cards <= #evil on the team)\\ninstead of per-card variables, and record legality is checked imperatively.\\nselfcheck.py compares the two implementations instance by instance.\\n\\"\\"\\"\\nfrom itertools import combinations\\n\\nO_EVIL = {\'morgana\', \'mordred\', \'oberon\', \'minion\'}\\nO_ORDER = (\'merlin\', \'percival\', \'servant\', \'morgana\', \'mordred\', \'oberon\', \'minion\')\\nO_EVIL_COUNT = {5: 2, 6: 2, 7: 3, 8: 3, 9: 3, 10: 4}\\nO_SIZES = {5: [2, 3, 2, 3, 3], 6: [2, 3, 4, 3, 4], 7: [2, 3, 3, 4, 4],\\n           8: [3, 4, 4, 5, 5], 9: [3, 4, 4, 5, 5], 10: [3, 4, 4, 5, 5]}\\n\\n\\ndef o_need(n, q):\\n    if q == 4 and n >= 7:\\n        return 2\\n    return 1\\n\\n\\ndef role_assignments(n, config):\\n    left = {r: config.get(r, 0) for r in O_ORDER}\\n    seats = [None] * n\\n\\n    def go(i):\\n        if i == n:\\n            yield tuple(seats)\\n            return\\n        for r in O_ORDER:\\n            if left[r]:\\n                left[r] -= 1\\n                seats[i] = r\\n                yield from go(i + 1)\\n                left[r] += 1\\n    if sum(left.values()) == n:\\n        yield from go(0)\\n\\n\\ndef record_violations(rec, variant):\\n    \\"\\"\\"Role-independent legality of the record; returns violated rule ids.\\"\\"\\"\\n    n, cfg, bad = rec[\'n\'], rec[\'config\'], []\\n    if n not in O_EVIL_COUNT:\\n        return [\'S-SIZE\']\\n    if sum(cfg.values()) != n:\\n        bad.append(\'S-TOTAL\')\\n    if sum(v for r, v in cfg.items() if r in O_EVIL) != O_EVIL_COUNT[n]:\\n        bad.append(\'S-COUNT\')\\n    if any(cfg.get(r, 0) > 1 for r in (\'merlin\', \'percival\', \'morgana\', \'mordred\', \'oberon\')):\\n        bad.append(\'S-UNIQUE\')\\n    wins = losses = 0\\n    last_leader = None\\n    quests = rec.get(\'quests\', [])\\n    for pos, qd in enumerate(quests):\\n        q = qd[\'q\']\\n        if q != pos + 1:\\n            bad.append(\'R-ORDER\')\\n            continue\\n        if wins >= 3 or losses >= 3:\\n            bad.append(\'G-END\')\\n        size = O_SIZES[n][q - 1]\\n        props = qd.get(\'proposals\', [])\\n        if len(props) > 5:\\n            bad.append(\'T-HAMMER\')\\n        for k, p in enumerate(props):\\n            team = p[\'team\']\\n            if len(team) != size:\\n                bad.append(\'T-SIZE\')\\n            if len(set(team)) != len(team) or any(not 0 <= x < n for x in team):\\n                bad.append(\'T-DISTINCT\')\\n            if p.get(\'approve\') is not None:\\n                yes = len(set(p[\'approve\']))\\n                if (2 * yes > n) != bool(p[\'approved\']):\\n                    bad.append(\'T-VOTE\')\\n            if variant.get(\'check_leader\', True) and last_leader is not None:\\n                if p[\'leader\'] != (last_leader + 1) % n:\\n                    bad.append(\'T-LEADER\')\\n            last_leader = p[\'leader\']\\n            if p[\'approved\'] and k != len(props) - 1:\\n                bad.append(\'T-LAST\')\\n        if qd.get(\'result\') is not None:\\n            if not props or not props[-1][\'approved\'] or sorted(props[-1][\'team\']) != sorted(qd[\'team\']):\\n                if props:\\n                    bad.append(\'T-LAST\')\\n            if len(qd[\'team\']) != size or len(set(qd[\'team\'])) != size:\\n                bad.append(\'Q-SIZE\')\\n            f = qd.get(\'fails\')\\n            if variant.get(\'fail_count_public\', True) and f is not None:\\n                if f < 0 or f > len(qd[\'team\']):\\n                    bad.append(\'Q-COUNT\')\\n                if (f >= o_need(n, q)) != (qd[\'result\'] == \'fail\'):\\n                    bad.append(\'Q-RESULT\')\\n            if qd[\'result\'] == \'success\':\\n                wins += 1\\n            else:\\n                losses += 1\\n        elif len(props) == 5 and not props[-1][\'approved\']:\\n            # fifth rejection: game over, evil wins (T-HAMMER)\\n            if pos != len(quests) - 1:\\n                bad.append(\'T-HAMMER\')\\n            end = rec.get(\'end\')\\n            if end and end.get(\'outcome\') not in (None, \'evil_win\'):\\n                bad.append(\'T-HAMMER\')\\n    # Lady of the Lake (optional rule): public part of each use\\n    resolved = {qd[\'q\'] for qd in quests if qd.get(\'result\') is not None}\\n    holders, prev_q = [], 0\\n    for k, use in enumerate(rec.get(\'lady\') or []):\\n        if use[\'q\'] not in (2, 3, 4) or use[\'q\'] <= prev_q or use[\'q\'] not in resolved:\\n            bad.append(\'L-WHEN\')\\n        prev_q = use[\'q\']\\n        if k == 0:\\n            first = quests[0][\'proposals\'][0][\'leader\'] if quests and quests[0].get(\'proposals\') else None\\n            if first is not None and use[\'holder\'] != (first - 1) % n:\\n                bad.append(\'L-FIRST\')\\n        elif use[\'holder\'] != rec[\'lady\'][k - 1][\'target\']:\\n            bad.append(\'L-CHAIN\')\\n        holders.append(use[\'holder\'])\\n        if use[\'target\'] in holders or not 0 <= use[\'target\'] < n:\\n            bad.append(\'L-PREV\')\\n    end = rec.get(\'end\')\\n    if end and end.get(\'outcome\') is not None:\\n        if losses >= 3 and end[\'outcome\'] != \'evil_win\':\\n            bad.append(\'G-OUTCOME\')\\n        if end.get(\'assassinated\') is not None and wins < 3:\\n            bad.append(\'G-ASSASSIN\')\\n    return sorted(set(bad))\\n\\n\\ndef _team_evil(roles, team):\\n    return sum(1 for i in team if roles[i] in O_EVIL)\\n\\n\\ndef world_ok(roles, rec, view, reveals, variant):\\n    n = rec[\'n\']\\n    for qd in rec.get(\'quests\', []):\\n        if qd.get(\'result\') is None:\\n            continue\\n        ev = _team_evil(roles, qd[\'team\'])\\n        f = qd.get(\'fails\')\\n        if variant.get(\'fail_count_public\', True) and f is not None:\\n            if f > ev:\\n                return False\\n        elif qd[\'result\'] == \'fail\' and ev < o_need(rec[\'n\'], qd[\'q\']):\\n            return False\\n    end = rec.get(\'end\')\\n    if end and end.get(\'assassinated\') is not None:\\n        hit = roles[end[\'assassinated\']] == \'merlin\'\\n        if (end[\'outcome\'] == \'evil_win\') != hit:\\n            return False\\n    if view is not None:\\n        p, r = view[\'player\'], view[\'role\']\\n        if roles[p] != r:\\n            return False\\n        seen = set(view.get(\'seen\') or [])\\n        if r == \'merlin\':\\n            exp = set()\\n            for j in range(n):\\n                if j == p or roles[j] not in O_EVIL or roles[j] == \'mordred\':\\n                    continue\\n                if roles[j] == \'oberon\' and not variant.get(\'merlin_sees_oberon\', True):\\n                    continue\\n                exp.add(j)\\n        elif r == \'percival\':\\n            exp = {j for j in range(n) if j != p and roles[j] in (\'merlin\', \'morgana\')}\\n        elif r in O_EVIL and r != \'oberon\':\\n            exp = {j for j in range(n) if j != p and roles[j] in O_EVIL and roles[j] != \'oberon\'}\\n        else:\\n            exp = set()  # servants and Oberon learn nothing at setup\\n        if exp != seen:\\n            return False\\n        for t, is_evil in view.get(\'lady\') or []:\\n            if (roles[t] in O_EVIL) != bool(is_evil):\\n                return False\\n    for t, is_evil in reveals or []:\\n        if (roles[t] in O_EVIL) != bool(is_evil):\\n            return False\\n    return True\\n\\n\\ndef claim_holds(c, roles, rec):\\n    t = c[\'t\']\\n    if t == \'evil\':\\n        return roles[c[\'p\']] in O_EVIL\\n    if t == \'good\':\\n        return roles[c[\'p\']] not in O_EVIL\\n    if t == \'role\':\\n        return roles[c[\'p\']] == c[\'r\']\\n    if t == \'atleast\':\\n        return _team_evil(roles, c[\'ps\']) >= c[\'k\']\\n    if t == \'exactly\':\\n        return _team_evil(roles, c[\'ps\']) == c[\'k\']\\n    if t == \'same\':\\n        a, b = c[\'ps\']\\n        return (roles[a] in O_EVIL) == (roles[b] in O_EVIL)\\n    if t == \'evil_set\':\\n        return {i for i, r in enumerate(roles) if r in O_EVIL} == set(c[\'ps\'])\\n    if t == \'not\':\\n        return not claim_holds(c[\'a\'], roles, rec)\\n    if t == \'and\':\\n        return all(claim_holds(x, roles, rec) for x in c[\'as\'])\\n    if t == \'or\':\\n        return any(claim_holds(x, roles, rec) for x in c[\'as\'])\\n    if t == \'fail_by\':\\n        return c[\'f\'] >= o_need(rec[\'n\'], c[\'q\'])\\n    if t == \'vote_pass\':\\n        return 2 * c[\'yes\'] > rec[\'n\']\\n    if t == \'team_size\':\\n        return O_SIZES[rec[\'n\']][c[\'q\'] - 1] == c[\'k\']\\n    raise ValueError(t)\\n\\n\\ndef worlds(inst):\\n    rec, view = inst[\'rec\'], inst.get(\'view\')\\n    variant = inst.get(\'variant\', {})\\n    if record_violations(rec, variant):\\n        return\\n    for roles in role_assignments(rec[\'n\'], rec[\'config\']):\\n        if world_ok(roles, rec, view, inst.get(\'reveals\'), variant):\\n            yield roles\\n\\n\\nclass Oracle:\\n    name = \'oracle\'\\n\\n    def classify(self, inst, claim):\\n        pos = neg = None\\n        for roles in worlds(inst):\\n            if claim_holds(claim, roles, inst[\'rec\']):\\n                pos = pos or roles\\n            else:\\n                neg = neg or roles\\n            if pos and neg:\\n                break\\n        if pos is None and neg is None:\\n            return {\'label\': \'inconsistent\', \'positive\': None, \'negative\': None}\\n        label = \'undetermined\' if pos and neg else (\'entailed\' if pos else \'contradicted\')\\n        return {\'label\': label, \'positive\': pos, \'negative\': neg}\\n\\n    def find_pair(self, inst, claim, agree):\\n        seen = {}\\n        for roles in worlds(inst):\\n            key = tuple(roles[j] in O_EVIL for j in agree)\\n            v = claim_holds(claim, roles, inst[\'rec\'])\\n            bucket = seen.setdefault(key, {})\\n            if (not v) in bucket:\\n                return bucket[not v], roles\\n            bucket.setdefault(v, roles)\\n        return None\\n\\n    def min_reveal_sets(self, inst, claim, candidates):\\n        ws = [(r, claim_holds(claim, r, inst[\'rec\'])) for r in worlds(inst)]\\n        if not ws:\\n            return None\\n        for k in range(len(candidates) + 1):\\n            found = []\\n            for S in combinations(candidates, k):\\n                groups = {}\\n                for r, v in ws:\\n                    groups.setdefault(tuple(r[j] in O_EVIL for j in S), set()).add(v)\\n                if all(len(g) == 1 for g in groups.values()):\\n                    found.append(list(S))\\n            if found:\\n                return found\\n        return []\\n\\n    def count_worlds(self, inst):\\n        return sum(1 for _ in worlds(inst))\\n\\n\\ndef determined_players(inst):\\n    \\"\\"\\"Per seat: \'evil\' / \'good\' / \'unknown\' over all consistent worlds (None if inconsistent).\\"\\"\\"\\n    seen = None\\n    for roles in worlds(inst):\\n        ev = [r in O_EVIL for r in roles]\\n        seen = [{e} for e in ev] if seen is None else [s | {e} for s, e in zip(seen, ev)]\\n    if seen is None:\\n        return None\\n    return [\'unknown\' if len(s) == 2 else (\'evil\' if True in s else \'good\') for s in seen]\\n\\n\\ndef possible_roles(inst, p):\\n    return sorted({roles[p] for roles in worlds(inst)})\\n", "avalon/smt.py": "\\"\\"\\"Z3 encoding of the Avalon rules (see RULES.md for the clause table).\\n\\nEvery rule instance is a tracked constraint whose tag starts with the clause\\nid (e.g. \'Q-GOOD@q3:5\' = \\"good players play Success\\", quest 3, seat 5), so\\nunsatisfiable cores read as lists of rule instances and recorded facts.\\nEach quest card is a Boolean variable (True = Fail); this is a different\\nformulation from oracle.py, which checks #Fail <= #evil on the team directly.\\n\\"\\"\\"\\nimport z3\\nfrom game import (ROLES, EVIL_ROLES, SPECIAL, EVIL_COUNT, TEAM_SIZE, DEFAULT_VARIANT,\\n                  fails_required)\\n\\nCODE = {r: i for i, r in enumerate(ROLES)}\\n\\n\\nclass SolverUnknown(RuntimeError):\\n    pass\\n\\n\\ndef _g(b):\\n    return z3.BoolVal(bool(b))\\n\\n\\nclass SMT:\\n    name = \'z3\'\\n\\n    def __init__(self, timeout_ms=60000):\\n        self.timeout = timeout_ms\\n        self.calls = 0\\n\\n    # ------------------------------------------------------------------ encoding\\n    def encode(self, inst, suffix=\'\'):\\n        rec, view = inst[\'rec\'], inst.get(\'view\')\\n        reveals = inst.get(\'reveals\') or []\\n        variant = {**DEFAULT_VARIANT, \'check_leader\': True, **inst.get(\'variant\', {})}\\n        n, cfg = rec[\'n\'], rec[\'config\']\\n        R = [z3.Int(f\'role{suffix}_{i}\') for i in range(n)]\\n        E = [z3.Or([x == CODE[r] for r in EVIL_ROLES]) for x in R]\\n        bg = [z3.And(x >= 0, x < len(ROLES)) for x in R]\\n        tr = []\\n\\n        def add(tag, expr):\\n            tr.append((tag, expr))\\n\\n        add(\'S-SIZE\', _g(n in EVIL_COUNT))\\n        if n not in EVIL_COUNT:\\n            return bg, tr, R, E\\n        add(\'S-TOTAL\', _g(sum(cfg.values()) == n))\\n        add(\'S-COUNT\', _g(sum(cfg.get(r, 0) for r in EVIL_ROLES) == EVIL_COUNT[n]))\\n        add(\'S-UNIQUE\', _g(all(cfg.get(r, 0) <= 1 for r in SPECIAL)))\\n        for r in ROLES:\\n            add(f\'S-DEAL[{r}]\', z3.Sum([z3.If(x == CODE[r], 1, 0) for x in R]) == cfg.get(r, 0))\\n\\n        quests = rec.get(\'quests\', [])\\n        end = rec.get(\'end\')\\n        wins = losses = 0\\n        last_leader = None\\n        for pos, qd in enumerate(quests):\\n            q = qd[\'q\']\\n            add(f\'R-ORDER@{pos + 1}\', _g(q == pos + 1))\\n            if q != pos + 1:\\n                continue\\n            add(f\'G-END@q{q}\', _g(wins < 3 and losses < 3))\\n            size = TEAM_SIZE[n][q - 1]\\n            props = qd.get(\'proposals\', [])\\n            add(f\'T-HAMMER@q{q}\', _g(len(props) <= 5))\\n            for k, p in enumerate(props):\\n                tag = f\'q{q}.{k + 1}\'\\n                team = p[\'team\']\\n                add(f\'T-SIZE@{tag}\', _g(len(team) == size))\\n                add(f\'T-DISTINCT@{tag}\', _g(len(set(team)) == len(team) and all(0 <= x < n for x in team)))\\n                if p.get(\'approve\') is not None:\\n                    add(f\'T-VOTE@{tag}\', _g((2 * len(set(p[\'approve\'])) > n) == bool(p[\'approved\'])))\\n                if variant[\'check_leader\'] and last_leader is not None:\\n                    add(f\'T-LEADER@{tag}\', _g(p[\'leader\'] == (last_leader + 1) % n))\\n                last_leader = p[\'leader\']\\n                if p[\'approved\'] and k != len(props) - 1:\\n                    add(f\'T-LAST@{tag}\', _g(False))\\n            if qd.get(\'result\') is not None:\\n                team = qd[\'team\']\\n                if props:\\n                    add(f\'T-LAST@q{q}\', _g(props[-1][\'approved\'] and sorted(props[-1][\'team\']) == sorted(team)))\\n                add(f\'Q-SIZE@q{q}\', _g(len(team) == size and len(set(team)) == size))\\n                cards = {i: z3.Bool(f\'card{suffix}_{q}_{i}\') for i in sorted(set(team)) if 0 <= i < n}\\n                for i, c in cards.items():\\n                    add(f\'Q-GOOD@q{q}:{i}\', z3.Implies(z3.Not(E[i]), z3.Not(c)))\\n                nf = z3.Sum([z3.If(c, 1, 0) for c in cards.values()]) if cards else z3.IntVal(0)\\n                f, need, failed = qd.get(\'fails\'), fails_required(n, q), qd[\'result\'] == \'fail\'\\n                if variant[\'fail_count_public\'] and f is not None:\\n                    add(f\'Q-COUNT@q{q}\', nf == f)\\n                    add(f\'Q-RESULT@q{q}\', _g((f >= need) == failed))\\n                else:\\n                    add(f\'Q-RESULT@q{q}\', (nf >= need) == failed)\\n                if failed:\\n                    losses += 1\\n                else:\\n                    wins += 1\\n            elif len(props) == 5 and not props[-1][\'approved\']:\\n                ok = pos == len(quests) - 1 and not (end and end.get(\'outcome\') not in (None, \'evil_win\'))\\n                add(f\'T-HAMMER@q{q}:end\', _g(ok))\\n\\n        resolved = {qd[\'q\'] for qd in quests if qd.get(\'result\') is not None}\\n        holders, prev_q = [], 0\\n        for k, use in enumerate(rec.get(\'lady\') or []):\\n            add(f\'L-WHEN@{k + 1}\', _g(use[\'q\'] in (2, 3, 4) and use[\'q\'] > prev_q and use[\'q\'] in resolved))\\n            prev_q = use[\'q\']\\n            if k == 0:\\n                first = quests[0][\'proposals\'][0][\'leader\'] if quests and quests[0].get(\'proposals\') else None\\n                if first is not None:\\n                    add(\'L-FIRST\', _g(use[\'holder\'] == (first - 1) % n))\\n            else:\\n                add(f\'L-CHAIN@{k + 1}\', _g(use[\'holder\'] == rec[\'lady\'][k - 1][\'target\']))\\n            holders.append(use[\'holder\'])\\n            add(f\'L-PREV@{k + 1}\', _g(use[\'target\'] not in holders and 0 <= use[\'target\'] < n))\\n\\n        if end and end.get(\'outcome\') is not None:\\n            add(\'G-OUTCOME\', _g(not (losses >= 3 and end[\'outcome\'] != \'evil_win\')))\\n            a = end.get(\'assassinated\')\\n            if a is not None:\\n                add(\'G-ASSASSIN\', _g(wins >= 3))\\n                if 0 <= a < n:\\n                    add(\'G-MERLIN-HIT\', (R[a] == CODE[\'merlin\']) == _g(end[\'outcome\'] == \'evil_win\'))\\n\\n        if view is not None:\\n            p, r = view[\'player\'], view[\'role\']\\n            seen = set(view.get(\'seen\') or [])\\n            add(\'V-SELF\', R[p] == CODE[r])\\n            add(\'K-SELF\', _g(p not in seen and all(0 <= j < n for j in seen)))\\n            for j in range(n):\\n                if j == p:\\n                    continue\\n                inS = _g(j in seen)\\n                if r == \'merlin\':\\n                    vis = z3.And(E[j], R[j] != CODE[\'mordred\'])\\n                    if not variant[\'merlin_sees_oberon\']:\\n                        vis = z3.And(vis, R[j] != CODE[\'oberon\'])\\n                    add(f\'K-MERLIN@{j}\', vis == inS)\\n                elif r == \'percival\':\\n                    add(f\'K-PERCIVAL@{j}\', z3.Or(R[j] == CODE[\'merlin\'], R[j] == CODE[\'morgana\']) == inS)\\n                elif r in EVIL_ROLES and r != \'oberon\':\\n                    add(f\'K-EVIL@{j}\', z3.And(E[j], R[j] != CODE[\'oberon\']) == inS)\\n                else:\\n                    add(f\'K-NONE@{j}\', _g(j not in seen))\\n            for t, ev in view.get(\'lady\') or []:\\n                add(f\'K-LADY@{t}\', E[t] == _g(ev))\\n        for t, ev in reveals:\\n            add(f\'X-REVEAL@{t}\', E[t] == _g(ev))\\n        return bg, tr, R, E\\n\\n    def claim(self, c, R, E, rec):\\n        t = c[\'t\']\\n        cnt = lambda ps: z3.Sum([z3.If(E[i], 1, 0) for i in ps]) if ps else z3.IntVal(0)\\n        if t == \'evil\':\\n            return E[c[\'p\']]\\n        if t == \'good\':\\n            return z3.Not(E[c[\'p\']])\\n        if t == \'role\':\\n            return R[c[\'p\']] == CODE[c[\'r\']]\\n        if t == \'atleast\':\\n            return cnt(c[\'ps\']) >= c[\'k\']\\n        if t == \'exactly\':\\n            return cnt(c[\'ps\']) == c[\'k\']\\n        if t == \'same\':\\n            a, b = c[\'ps\']\\n            return E[a] == E[b]\\n        if t == \'evil_set\':\\n            S = set(c[\'ps\'])\\n            return z3.And([E[i] == _g(i in S) for i in range(len(R))])\\n        if t == \'not\':\\n            return z3.Not(self.claim(c[\'a\'], R, E, rec))\\n        if t == \'and\':\\n            return z3.And([self.claim(x, R, E, rec) for x in c[\'as\']])\\n        if t == \'or\':\\n            return z3.Or([self.claim(x, R, E, rec) for x in c[\'as\']])\\n        if t == \'fail_by\':\\n            return _g(c[\'f\'] >= fails_required(rec[\'n\'], c[\'q\']))\\n        if t == \'vote_pass\':\\n            return _g(2 * c[\'yes\'] > rec[\'n\'])\\n        if t == \'team_size\':\\n            return _g(TEAM_SIZE[rec[\'n\']][c[\'q\'] - 1] == c[\'k\'])\\n        raise ValueError(t)\\n\\n    # ------------------------------------------------------------------ solving\\n    def _solver(self):\\n        s = z3.Solver()\\n        s.set(timeout=self.timeout)\\n        return s\\n\\n    def _check(self, s, *assume):\\n        self.calls += 1\\n        r = s.check(*assume)\\n        if r == z3.unknown:\\n            raise SolverUnknown(s.reason_unknown())\\n        return r\\n\\n    @staticmethod\\n    def _decode(m, R):\\n        return tuple(ROLES[m.eval(x, model_completion=True).as_long()] for x in R)\\n\\n    def classify(self, inst, claim, explain=False):\\n        bg, tr, R, E = self.encode(inst)\\n        s = self._solver()\\n        s.add(*bg)\\n        s.add(*[e for _, e in tr])\\n        if self._check(s) == z3.unsat:\\n            out = {\'label\': \'inconsistent\', \'positive\': None, \'negative\': None}\\n            if explain:\\n                out[\'core\'] = self.core(bg, tr)\\n            return out\\n        cz = self.claim(claim, R, E, inst[\'rec\'])\\n        wit = []\\n        for side in (cz, z3.Not(cz)):\\n            s.push()\\n            s.add(side)\\n            wit.append(self._decode(s.model(), R) if self._check(s) == z3.sat else None)\\n            s.pop()\\n        pos, neg = wit\\n        label = \'undetermined\' if pos and neg else (\'entailed\' if pos else \'contradicted\')\\n        out = {\'label\': label, \'positive\': pos, \'negative\': neg}\\n        if explain and label == \'entailed\':\\n            out[\'core\'] = self.core(bg, tr, z3.Not(cz))\\n        if explain and label == \'contradicted\':\\n            out[\'core\'] = self.core(bg, tr, cz)\\n        return out\\n\\n    def core(self, bg, tr, extra=None):\\n        \\"\\"\\"Subset-minimal set of tracked rule instances (deletion-based).\\"\\"\\"\\n        s = self._solver()\\n        s.add(*bg)\\n        if extra is not None:\\n            s.add(extra)\\n        lits = [z3.Bool(f\'__t{i}\') for i in range(len(tr))]\\n        for lit, (_, e) in zip(lits, tr):\\n            s.add(z3.Implies(lit, e))\\n        if self._check(s, *lits) != z3.unsat:\\n            raise AssertionError(\'core of a satisfiable query\')\\n        index = {str(l): i for i, l in enumerate(lits)}\\n        cur = sorted(index[str(x)] for x in s.unsat_core())\\n        i = 0\\n        while i < len(cur):\\n            trial = cur[:i] + cur[i + 1:]\\n            if self._check(s, *[lits[j] for j in trial]) == z3.unsat:\\n                cur = trial\\n            else:\\n                i += 1\\n        return [tr[j][0] for j in cur]\\n\\n    def find_pair(self, inst, claim, agree):\\n        bga, tra, Ra, Ea = self.encode(inst, \'_a\')\\n        bgb, trb, Rb, Eb = self.encode(inst, \'_b\')\\n        s = self._solver()\\n        s.add(*bga, *bgb, *[e for _, e in tra], *[e for _, e in trb])\\n        s.add(self.claim(claim, Ra, Ea, inst[\'rec\']) != self.claim(claim, Rb, Eb, inst[\'rec\']))\\n        s.add(*[Ea[j] == Eb[j] for j in agree])\\n        if self._check(s) == z3.unsat:\\n            return None\\n        m = s.model()\\n        return self._decode(m, Ra), self._decode(m, Rb)\\n\\n    def determined_players(self, inst):\\n        \\"\\"\\"For each seat: \'evil\' / \'good\' / \'unknown\' (or None if inconsistent).\\"\\"\\"\\n        bg, tr, R, E = self.encode(inst)\\n        s = self._solver()\\n        s.add(*bg, *[e for _, e in tr])\\n        if self._check(s) == z3.unsat:\\n            return None\\n        out = []\\n        for e in E:\\n            s.push(); s.add(e); can_evil = self._check(s) == z3.sat; s.pop()\\n            s.push(); s.add(z3.Not(e)); can_good = self._check(s) == z3.sat; s.pop()\\n            out.append(\'unknown\' if can_evil and can_good else (\'evil\' if can_evil else \'good\'))\\n        return out\\n\\n    def possible_roles(self, inst, p):\\n        bg, tr, R, E = self.encode(inst)\\n        s = self._solver()\\n        s.add(*bg, *[e for _, e in tr])\\n        out = []\\n        for r in ROLES:\\n            s.push()\\n            s.add(R[p] == CODE[r])\\n            if self._check(s) == z3.sat:\\n                out.append(r)\\n            s.pop()\\n        return out\\n", "avalon/search.py": "\\"\\"\\"Minimal information completion for Avalon claims (backend-agnostic).\\n\\nA *reveal* tells the alignment (good/evil) of one player, exactly what a Lady\\nof the Lake check gives its holder.  For an undetermined claim C we ask:\\n\\n  one-shot   : the smallest set S of players whose alignments, whatever they\\n               turn out to be, always decide C.  S is sufficient iff no two\\n               consistent worlds agree on the alignments in S but disagree on C\\n               (two-world test, backend.find_pair).\\n               - naive : enumerate subsets by increasing size\\n               - cegar : counterexample-guided implicit hitting sets\\n  adaptive   : the minimax number of sequential reveals, where each next\\n               player is chosen after seeing earlier answers (a game against\\n               an adversarial world; answers must stay consistent).\\n\\nBackends implement classify(inst, claim) and find_pair(inst, claim, agree).\\n\\"\\"\\"\\nimport time\\nfrom itertools import combinations\\n\\nEVIL = {\'morgana\', \'mordred\', \'oberon\', \'minion\'}\\nDECIDED = (\'entailed\', \'contradicted\')\\n\\n\\ndef _ev(roles, j):\\n    return roles[j] in EVIL\\n\\n\\ndef min_hitting_set(sets, costs):\\n    best = [float(\'inf\'), None]\\n    sets = [frozenset(s) for s in sets]\\n    if any(not s for s in sets):\\n        return None\\n\\n    def rec(chosen, cost):\\n        if cost >= best[0]:\\n            return\\n        unhit = None\\n        for s in sets:\\n            if not (s & chosen) and (unhit is None or len(s) < len(unhit)):\\n                unhit = s\\n        if unhit is None:\\n            best[0], best[1] = cost, chosen\\n            return\\n        for f in sorted(unhit, key=lambda x: (costs[x], x)):\\n            rec(chosen | {f}, cost + costs[f])\\n    rec(frozenset(), 0)\\n    return None if best[1] is None else sorted(best[1])\\n\\n\\ndef candidates(inst):\\n    known = {t for t, _ in inst.get(\'reveals\') or []}\\n    view = inst.get(\'view\')\\n    if view is not None:\\n        known.add(view[\'player\'])\\n        known |= {t for t, _ in view.get(\'lady\') or []}\\n    return [j for j in range(inst[\'rec\'][\'n\']) if j not in known]\\n\\n\\ndef min_reveal_naive(backend, inst, claim, cands=None, deadline=None):\\n    cands = candidates(inst) if cands is None else cands\\n    checks, t0 = 0, time.perf_counter()\\n    for k in range(len(cands) + 1):\\n        found = []\\n        for S in combinations(cands, k):\\n            if deadline and time.perf_counter() - t0 > deadline:\\n                return {\'status\': \'timeout\', \'checks\': checks, \'size_reached\': k}\\n            checks += 1\\n            if backend.find_pair(inst, claim, list(S)) is None:\\n                found.append(list(S))\\n        if found:\\n            return {\'status\': \'ok\', \'sets\': found, \'size\': k, \'checks\': checks}\\n    return {\'status\': \'none\', \'checks\': checks}\\n\\n\\ndef min_reveal_cegar(backend, inst, claim, cands=None, costs=None, generalize=True):\\n    cands = candidates(inst) if cands is None else cands\\n    costs = costs or {j: 1 for j in cands}\\n    D, calls, rounds = [], 0, 0\\n    while True:\\n        rounds += 1\\n        H = min_hitting_set(D, costs)\\n        if H is None:\\n            return {\'status\': \'none\', \'calls\': calls, \'rounds\': rounds}\\n        pair = backend.find_pair(inst, claim, H)\\n        calls += 1\\n        if pair is None:\\n            return {\'status\': \'ok\', \'set\': H, \'size\': len(H), \'calls\': calls, \'rounds\': rounds,\\n                    \'cuts\': [sorted(d) for d in D]}\\n        x, y = pair\\n        diff = [j for j in cands if j not in H and _ev(x, j) != _ev(y, j)]\\n        if generalize:\\n            agree = list(H)\\n            for j in list(diff):\\n                if j not in diff:\\n                    continue\\n                trial = backend.find_pair(inst, claim, agree + [j])\\n                calls += 1\\n                if trial is not None:\\n                    agree.append(j)\\n                    x, y = trial\\n                    diff = [g for g in cands if g not in agree and _ev(x, g) != _ev(y, g)]\\n        if not diff:\\n            return {\'status\': \'none\', \'calls\': calls, \'rounds\': rounds}\\n        D.append(frozenset(diff))\\n\\n\\ndef adaptive_depth(backend, inst, claim, cands=None, limit=None):\\n    \\"\\"\\"Worst-case number of sequential reveals needed to decide the claim.\\n\\n    Returns (depth, first_choice); depth is None if no reveal strategy over\\n    `cands` can decide the claim.\\"\\"\\"\\n    cands = candidates(inst) if cands is None else cands\\n    memo = {}\\n    base = list(inst.get(\'reveals\') or [])\\n\\n    def label(extra):\\n        return backend.classify({**inst, \'reveals\': base + list(extra)}, claim)[\'label\']\\n\\n    def solve(extra):\\n        key = frozenset(extra)\\n        if key in memo:\\n            return memo[key]\\n        lab = label(extra)\\n        if lab in DECIDED or lab == \'inconsistent\':\\n            memo[key] = (0, None)\\n            return memo[key]\\n        used = {t for t, _ in extra}\\n        best = (None, None)\\n        for j in cands:\\n            if j in used:\\n                continue\\n            worst = 0\\n            for v in (False, True):\\n                nxt = tuple(sorted(extra + ((j, v),)))\\n                if label(nxt) == \'inconsistent\':\\n                    continue\\n                d, _ = solve(nxt)\\n                if d is None:\\n                    worst = None\\n                    break\\n                worst = max(worst, d + 1)\\n            if worst is not None and (best[0] is None or worst < best[0]):\\n                best = (worst, j)\\n                if limit is not None and worst <= 1:\\n                    break\\n        memo[key] = best\\n        return best\\n\\n    return solve(tuple())\\n", "avalon/sim.py": "\\"\\"\\"Random legal Avalon games (for cross-checking and synthetic tasks).\\"\\"\\"\\nimport random\\nfrom game import EVIL_ROLES, EVIL_COUNT, TEAM_SIZE, fails_required\\n\\nCONFIGS = {\\n    5: [{\'merlin\': 1, \'servant\': 2, \'minion\': 2},\\n        {\'merlin\': 1, \'percival\': 1, \'servant\': 1, \'morgana\': 1, \'minion\': 1}],\\n    6: [{\'merlin\': 1, \'percival\': 1, \'servant\': 2, \'morgana\': 1, \'minion\': 1},\\n        {\'merlin\': 1, \'servant\': 3, \'minion\': 2}],\\n    7: [{\'merlin\': 1, \'percival\': 1, \'servant\': 2, \'morgana\': 1, \'mordred\': 1, \'minion\': 1},\\n        {\'merlin\': 1, \'percival\': 1, \'servant\': 2, \'morgana\': 1, \'oberon\': 1, \'minion\': 1}],\\n    8: [{\'merlin\': 1, \'percival\': 1, \'servant\': 3, \'morgana\': 1, \'mordred\': 1, \'minion\': 1}],\\n    9: [{\'merlin\': 1, \'percival\': 1, \'servant\': 4, \'morgana\': 1, \'mordred\': 1, \'minion\': 1}],\\n    10: [{\'merlin\': 1, \'percival\': 1, \'servant\': 4, \'morgana\': 1, \'mordred\': 1, \'oberon\': 1, \'minion\': 1}],\\n}\\n\\n\\ndef deal(rng, config):\\n    roles = [r for r, c in config.items() for _ in range(c)]\\n    rng.shuffle(roles)\\n    return tuple(roles)\\n\\n\\ndef play(rng, n=None, config=None, p_fail=0.65, p_approve=0.5, lady=None):\\n    n = n or rng.choice(sorted(CONFIGS))\\n    config = config or rng.choice(CONFIGS[n])\\n    roles = deal(rng, config)\\n    evil = [r in EVIL_ROLES for r in roles]\\n    leader = rng.randrange(n)\\n    first_leader = leader\\n    quests, wins, losses = [], 0, 0\\n    end = None\\n    for q in range(1, 6):\\n        size = TEAM_SIZE[n][q - 1]\\n        props = []\\n        team = None\\n        for k in range(5):\\n            t = sorted(rng.sample(range(n), size))\\n            approve = sorted(i for i in range(n) if rng.random() < p_approve)\\n            ok = 2 * len(approve) > n\\n            if k == 4 and not ok:\\n                pass\\n            props.append({\'leader\': leader, \'team\': t, \'approve\': approve, \'approved\': ok})\\n            leader = (leader + 1) % n\\n            if ok:\\n                team = t\\n                break\\n        if team is None:  # five rejections: evil wins\\n            quests.append({\'q\': q, \'proposals\': props, \'team\': [], \'fails\': None, \'result\': None})\\n            end = {\'assassinated\': None, \'outcome\': \'evil_win\'}\\n            break\\n        fails = sum(1 for i in team if evil[i] and rng.random() < p_fail)\\n        failed = fails >= fails_required(n, q)\\n        quests.append({\'q\': q, \'proposals\': props, \'team\': team, \'fails\': fails,\\n                       \'result\': \'fail\' if failed else \'success\'})\\n        losses += failed\\n        wins += not failed\\n        if losses == 3:\\n            end = {\'assassinated\': None, \'outcome\': \'evil_win\'}\\n            break\\n        if wins == 3:\\n            target = rng.choice([i for i in range(n) if not evil[i]])\\n            end = {\'assassinated\': target,\\n                   \'outcome\': \'evil_win\' if roles[target] == \'merlin\' else \'good_win\'}\\n            break\\n    rec = {\'n\': n, \'config\': dict(config), \'quests\': quests, \'end\': end}\\n    if lady:\\n        holder = (first_leader - 1) % n\\n        used, uses = {holder}, []\\n        for qd in quests:\\n            if qd[\'q\'] in (2, 3, 4) and qd.get(\'result\') is not None:\\n                choices = [i for i in range(n) if i not in used]\\n                t = rng.choice(choices)\\n                uses.append({\'q\': qd[\'q\'], \'holder\': holder, \'target\': t})\\n                used.add(t)\\n                holder = t\\n        rec[\'lady\'] = uses\\n    return rec, roles\\n\\n\\ndef true_view(roles, p, variant=None, rec=None):\\n    variant = variant or {}\\n    n, r = len(roles), roles[p]\\n    if r == \'merlin\':\\n        seen = [j for j in range(n) if j != p and roles[j] in EVIL_ROLES and roles[j] != \'mordred\'\\n                and (variant.get(\'merlin_sees_oberon\', True) or roles[j] != \'oberon\')]\\n    elif r == \'percival\':\\n        seen = [j for j in range(n) if j != p and roles[j] in (\'merlin\', \'morgana\')]\\n    elif r in EVIL_ROLES and r != \'oberon\':\\n        seen = [j for j in range(n) if j != p and roles[j] in EVIL_ROLES and roles[j] != \'oberon\']\\n    else:\\n        seen = []\\n    lady = []\\n    if rec is not None:\\n        for use in rec.get(\'lady\') or []:\\n            if use[\'holder\'] == p:\\n                lady.append([use[\'target\'], roles[use[\'target\']] in EVIL_ROLES])\\n    return {\'player\': p, \'role\': r, \'seen\': seen, \'lady\': lady}\\n\\n\\ndef prefix(rec, k):\\n    \\"\\"\\"Public record after the first k quests (no end information).\\"\\"\\"\\n    return {**rec, \'quests\': rec[\'quests\'][:k], \'end\': None,\\n            \'lady\': [u for u in rec.get(\'lady\') or [] if u[\'q\'] <= k] if \'lady\' in rec else None}\\n\\n\\ndef random_claim(rng, n):\\n    kind = rng.choice([\'evil\', \'good\', \'role\', \'atleast\', \'exactly\', \'same\', \'and\', \'not\'])\\n    p = rng.randrange(n)\\n    if kind in (\'evil\', \'good\'):\\n        return {\'t\': kind, \'p\': p}\\n    if kind == \'role\':\\n        return {\'t\': \'role\', \'p\': p, \'r\': rng.choice([\'merlin\', \'percival\', \'morgana\', \'mordred\', \'servant\', \'minion\', \'oberon\'])}\\n    if kind in (\'atleast\', \'exactly\'):\\n        ps = sorted(rng.sample(range(n), rng.randint(2, min(4, n))))\\n        return {\'t\': kind, \'ps\': ps, \'k\': rng.randint(0, len(ps))}\\n    if kind == \'same\':\\n        a, b = rng.sample(range(n), 2)\\n        return {\'t\': \'same\', \'ps\': [a, b]}\\n    if kind == \'and\':\\n        a, b = rng.sample(range(n), 2)\\n        return {\'t\': \'and\', \'as\': [{\'t\': \'evil\', \'p\': a}, {\'t\': \'good\', \'p\': b}]}\\n    return {\'t\': \'not\', \'a\': {\'t\': \'role\', \'p\': p, \'r\': \'merlin\'}}\\n\\n\\ndef corrupt(rng, rec):\\n    \\"\\"\\"Introduce one rule violation (for inconsistent-record tests).\\"\\"\\"\\n    import copy\\n    r = copy.deepcopy(rec)\\n    done = [qd for qd in r[\'quests\'] if qd.get(\'result\') is not None]\\n    if not done:\\n        r[\'config\'] = {**r[\'config\'], \'minion\': r[\'config\'].get(\'minion\', 0) + 1}\\n        return r, \'S-TOTAL\'\\n    qd = rng.choice(done)\\n    kind = rng.choice([\'fails_over\', \'size\', \'result\', \'vote\'])\\n    if kind == \'fails_over\':\\n        qd[\'fails\'] = len(qd[\'team\']) + 1\\n        return r, \'Q-COUNT\'\\n    if kind == \'size\':\\n        extra = [i for i in range(r[\'n\']) if i not in qd[\'team\']]\\n        qd[\'team\'] = sorted(qd[\'team\'] + extra[:1])\\n        if qd[\'proposals\']:\\n            qd[\'proposals\'][-1][\'team\'] = list(qd[\'team\'])\\n        return r, \'Q-SIZE\'\\n    if kind == \'result\':\\n        qd[\'result\'] = \'success\' if qd[\'result\'] == \'fail\' else \'fail\'\\n        return r, \'Q-RESULT\'\\n    p = qd[\'proposals\'][-1]\\n    p[\'approve\'] = p[\'approve\'][: r[\'n\'] // 2]\\n    return r, \'T-VOTE\'\\n", "avalon/cases.py": "\\"\\"\\"Hand-written test scenarios: natural-language situation, formalisation,\\na claimed conclusion, the expected verdict (derived by hand from the\\nrulebook) and the clauses exercised.  Seats are written 1-based in the text\\nand 0-based in the formalisation (helper S).\\n\\nVerdicts: entailed = the claim must hold (correct answer), contradicted =\\nthe claim cannot hold (wrong answer), undetermined = the record does not\\nsettle it (unsupported answer), inconsistent = the record itself breaks the\\nrules.\\n\\"\\"\\"\\n\\nC5 = {\'merlin\': 1, \'servant\': 2, \'minion\': 2}\\nC6 = {\'merlin\': 1, \'percival\': 1, \'servant\': 2, \'morgana\': 1, \'minion\': 1}\\nC7 = {\'merlin\': 1, \'percival\': 1, \'servant\': 2, \'morgana\': 1, \'mordred\': 1, \'minion\': 1}\\nC7O = {\'merlin\': 1, \'percival\': 1, \'servant\': 2, \'morgana\': 1, \'oberon\': 1, \'minion\': 1}\\nC8 = {\'merlin\': 1, \'percival\': 1, \'servant\': 3, \'morgana\': 1, \'mordred\': 1, \'minion\': 1}\\nC10 = {\'merlin\': 1, \'percival\': 1, \'servant\': 4, \'morgana\': 1, \'mordred\': 1, \'oberon\': 1, \'minion\': 1}\\n\\n\\ndef S(*seats):\\n    return [s - 1 for s in seats]\\n\\n\\ndef Q(q, team, fails, result=None, leader=None, n=None, proposals=None):\\n    \\"\\"\\"A completed quest; result defaults to the rulebook outcome.\\"\\"\\"\\n    from game import fails_required\\n    if result is None:\\n        result = \'fail\' if fails is not None and fails >= fails_required(n, q) else \'success\'\\n    d = {\'q\': q, \'team\': S(*team), \'fails\': fails, \'result\': result}\\n    if proposals is not None:\\n        d[\'proposals\'] = proposals\\n    elif leader is not None:\\n        d[\'proposals\'] = [{\'leader\': leader - 1, \'team\': S(*team), \'approved\': True, \'approve\': None}]\\n    else:\\n        d[\'proposals\'] = []\\n    return d\\n\\n\\ndef P(leader, team, approved, approve=None):\\n    return {\'leader\': leader - 1, \'team\': S(*team), \'approved\': approved,\\n            \'approve\': None if approve is None else S(*approve)}\\n\\n\\ndef rec(n, cfg, quests=(), end=None, lady=None):\\n    r = {\'n\': n, \'config\': cfg, \'quests\': list(quests), \'end\': end}\\n    if lady:\\n        r[\'lady\'] = [{\'q\': q, \'holder\': h - 1, \'target\': t - 1} for q, h, t in lady]\\n    return r\\n\\n\\ndef view(seat, role, seen=(), lady=()):\\n    return {\'player\': seat - 1, \'role\': role, \'seen\': S(*seen),\\n            \'lady\': [[t - 1, ev] for t, ev in lady]}\\n\\n\\nEVIL = lambda s: {\'t\': \'evil\', \'p\': s - 1}\\nGOOD = lambda s: {\'t\': \'good\', \'p\': s - 1}\\nROLE = lambda s, r: {\'t\': \'role\', \'p\': s - 1, \'r\': r}\\nAND = lambda *a: {\'t\': \'and\', \'as\': list(a)}\\nNOT = lambda a: {\'t\': \'not\', \'a\': a}\\nATLEAST = lambda seats, k: {\'t\': \'atleast\', \'ps\': S(*seats), \'k\': k}\\nEXACTLY = lambda seats, k: {\'t\': \'exactly\', \'ps\': S(*seats), \'k\': k}\\nEVILSET = lambda *seats: {\'t\': \'evil_set\', \'ps\': S(*seats)}\\n\\nALL7 = (1, 2, 3, 4, 5, 6, 7)\\n\\nCASES = [\\n    # ---------------- setup tables ----------------\\n    dict(id=\'T01\', cat=\'simple\', rules=[\'S-COUNT\', \'S-DEAL\'],\\n         text=\'7人局，角色为梅林、派西维尔、忠臣×2、莫甘娜、莫德雷德、爪牙。问：本局有几名坏人？答：3名。\',\\n         inst={\'rec\': rec(7, C7)}, claim=EXACTLY(ALL7, 3), expect=\'entailed\'),\\n    dict(id=\'T02\', cat=\'simple\', rules=[\'S-COUNT\', \'S-DEAL\'],\\n         text=\'同上。答：2名坏人。\', inst={\'rec\': rec(7, C7)}, claim=EXACTLY(ALL7, 2), expect=\'contradicted\'),\\n    dict(id=\'T03\', cat=\'inconsistent\', rules=[\'S-COUNT\'],\\n         text=\'6人局却配置了莫甘娜、莫德雷德、爪牙三名坏人。问：1号是好人吗？答：是。\',\\n         inst={\'rec\': rec(6, {\'merlin\': 1, \'servant\': 2, \'morgana\': 1, \'mordred\': 1, \'minion\': 1})},\\n         claim=GOOD(1), expect=\'inconsistent\'),\\n    dict(id=\'T04\', cat=\'simple\', rules=[\'T-SIZE\'],\\n         text=\'8人局第1个任务需要几人出队？答：3人。\', inst={\'rec\': rec(8, C8)},\\n         claim={\'t\': \'team_size\', \'q\': 1, \'k\': 3}, expect=\'entailed\'),\\n    dict(id=\'T05\', cat=\'boundary\', rules=[\'T-SIZE\'],\\n         text=\'5人局第3个任务需要几人出队？答：3人。\', inst={\'rec\': rec(5, C5)},\\n         claim={\'t\': \'team_size\', \'q\': 3, \'k\': 3}, expect=\'contradicted\'),\\n    dict(id=\'T06\', cat=\'inconsistent\', rules=[\'Q-SIZE\'],\\n         text=\'5人局第1个任务由1、2、3号三人执行，0张失败票。问：4号是好人吗？答：是。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2, 3), 0, n=5)])}, claim=GOOD(4), expect=\'inconsistent\'),\\n    # ---------------- two-fail rule ----------------\\n    dict(id=\'T07\', cat=\'boundary\', rules=[\'Q-TWOFAIL\'],\\n         text=\'7人局第4个任务翻开1张失败票。问：任务失败了吗？答：失败。\', inst={\'rec\': rec(7, C7)},\\n         claim={\'t\': \'fail_by\', \'q\': 4, \'f\': 1}, expect=\'contradicted\'),\\n    dict(id=\'T08\', cat=\'boundary\', rules=[\'Q-TWOFAIL\'],\\n         text=\'6人局第4个任务翻开1张失败票。问：任务失败了吗？答：失败。\', inst={\'rec\': rec(6, C6)},\\n         claim={\'t\': \'fail_by\', \'q\': 4, \'f\': 1}, expect=\'entailed\'),\\n    dict(id=\'T09\', cat=\'boundary\', rules=[\'Q-TWOFAIL\'],\\n         text=\'7人局第4个任务翻开2张失败票。问：任务失败了吗？答：失败。\', inst={\'rec\': rec(7, C7)},\\n         claim={\'t\': \'fail_by\', \'q\': 4, \'f\': 2}, expect=\'entailed\'),\\n    dict(id=\'T10\', cat=\'boundary\', rules=[\'Q-TWOFAIL\'],\\n         text=\'7人局第3个任务翻开1张失败票。问：任务失败了吗？答：失败。\', inst={\'rec\': rec(7, C7)},\\n         claim={\'t\': \'fail_by\', \'q\': 3, \'f\': 1}, expect=\'entailed\'),\\n    dict(id=\'T11\', cat=\'inconsistent\', rules=[\'Q-RESULT\', \'Q-TWOFAIL\'],\\n         text=\'7人局记录：前三个任务正常，第4个任务4人出队、1张失败票，却被记为失败。问：1号是坏人吗？答：是。\',\\n         inst={\'rec\': rec(7, C7, [Q(1, (1, 2), 0, n=7), Q(2, (1, 2, 3), 0, n=7), Q(3, (1, 2, 4), 1, n=7),\\n                                   Q(4, (1, 2, 3, 4), 1, result=\'fail\', n=7)])},\\n         claim=EVIL(1), expect=\'inconsistent\'),\\n    # ---------------- voting ----------------\\n    dict(id=\'T12\', cat=\'boundary\', rules=[\'T-VOTE\'],\\n         text=\'6人局，提案得到3票赞成。问：提案通过了吗？答：通过。\', inst={\'rec\': rec(6, C6)},\\n         claim={\'t\': \'vote_pass\', \'yes\': 3}, expect=\'contradicted\'),\\n    dict(id=\'T13\', cat=\'boundary\', rules=[\'T-VOTE\'],\\n         text=\'7人局，提案得到4票赞成。问：提案通过了吗？答：通过。\', inst={\'rec\': rec(7, C7)},\\n         claim={\'t\': \'vote_pass\', \'yes\': 4}, expect=\'entailed\'),\\n    dict(id=\'T14\', cat=\'inconsistent\', rules=[\'T-VOTE\'],\\n         text=\'8人局第1轮：队长1号提名1、2、3号，赞成者为1、2、3、4号，记为通过。问：5号是好人吗？答：是。\',\\n         inst={\'rec\': rec(8, C8, [{\'q\': 1, \'team\': S(1, 2, 3), \'fails\': 0, \'result\': \'success\',\\n                                    \'proposals\': [P(1, (1, 2, 3), True, approve=(1, 2, 3, 4))]}])},\\n         claim=GOOD(5), expect=\'inconsistent\'),\\n    dict(id=\'T15\', cat=\'inconsistent\', rules=[\'T-HAMMER\'],\\n         text=\'5人局第1轮连续5次提案都被否决，但随后第2轮仍继续进行。问：1号是好人吗？答：是。\',\\n         inst={\'rec\': rec(5, C5, [{\'q\': 1, \'team\': [], \'fails\': None, \'result\': None,\\n                                    \'proposals\': [P(k, (k, k % 5 + 1), False) for k in range(1, 6)]},\\n                                   {\'q\': 2, \'team\': S(1, 2, 3), \'fails\': 0, \'result\': \'success\',\\n                                    \'proposals\': [P(1, (1, 2, 3), True)]}])},\\n         claim=GOOD(1), expect=\'inconsistent\'),\\n    dict(id=\'T16\', cat=\'inconsistent\', rules=[\'T-LEADER\'],\\n         text=\'6人局第1轮：1号提案被否决后，下一次提案的队长是3号。问：1号是好人吗？答：是。\',\\n         inst={\'rec\': rec(6, C6, [{\'q\': 1, \'team\': S(3, 4), \'fails\': 0, \'result\': \'success\',\\n                                    \'proposals\': [P(1, (1, 2), False), P(3, (3, 4), True)]}])},\\n         claim=GOOD(1), expect=\'inconsistent\'),\\n    # ---------------- deductions from quests ----------------\\n    dict(id=\'T17\', cat=\'simple\', rules=[\'Q-GOOD\', \'Q-COUNT\'],\\n         text=\'5人局第1个任务由1、2号执行，翻开2张失败票。问：1号和2号都是坏人吗？答：是。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2), 2, n=5)])}, claim=AND(EVIL(1), EVIL(2)), expect=\'entailed\'),\\n    dict(id=\'T18\', cat=\'combination\', rules=[\'Q-GOOD\', \'Q-COUNT\', \'S-COUNT\'],\\n         text=\'同上。问：3号是好人吗？答：是（坏人只有2名，已全部在队中）。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2), 2, n=5)])}, claim=GOOD(3), expect=\'entailed\'),\\n    dict(id=\'T19\', cat=\'simple\', rules=[\'Q-GOOD\', \'Q-COUNT\'],\\n         text=\'5人局第1个任务由1、2号执行，翻开1张失败票。问：队里恰好有一个坏人吗？答：是。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2), 1, n=5)])}, claim=EXACTLY((1, 2), 1), expect=\'undetermined\'),\\n    dict(id=\'T20\', cat=\'simple\', rules=[\'Q-GOOD\', \'Q-COUNT\'],\\n         text=\'同上。问：队里至少有一个坏人吗？答：是。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2), 1, n=5)])}, claim=ATLEAST((1, 2), 1), expect=\'entailed\'),\\n    dict(id=\'T21\', cat=\'simple\', rules=[\'Q-GOOD\'],\\n         text=\'5人局第1个任务由1、2号执行，0张失败票，任务成功。问：1号和2号都是好人吗？答：是。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2), 0, n=5)])}, claim=AND(GOOD(1), GOOD(2)), expect=\'undetermined\'),\\n    dict(id=\'T22\', cat=\'combination\', rules=[\'Q-COUNT\', \'S-COUNT\'],\\n         text=\'7人局第2个任务由1、2、3号执行，翻开3张失败票。问：4到7号都是好人吗？答：是。\',\\n         inst={\'rec\': rec(7, C7, [Q(1, (4, 5), 0, n=7), Q(2, (1, 2, 3), 3, n=7)])},\\n         claim=AND(*[GOOD(s) for s in (4, 5, 6, 7)]), expect=\'entailed\'),\\n    dict(id=\'T23\', cat=\'combination\', rules=[\'Q-COUNT\', \'S-COUNT\'],\\n         text=\'6人局：第1个任务（1、2号）1张失败票；第2个任务（3、4、5号）1张失败票。问：6号是好人吗？答：是。\',\\n         inst={\'rec\': rec(6, C6, [Q(1, (1, 2), 1, n=6), Q(2, (3, 4, 5), 1, n=6)])},\\n         claim=GOOD(6), expect=\'entailed\'),\\n    dict(id=\'T24\', cat=\'combination\', rules=[\'Q-COUNT\', \'S-COUNT\'],\\n         text=\'同上。问：1号是坏人吗？答：是。\',\\n         inst={\'rec\': rec(6, C6, [Q(1, (1, 2), 1, n=6), Q(2, (3, 4, 5), 1, n=6)])},\\n         claim=EVIL(1), expect=\'undetermined\'),\\n    dict(id=\'T25\', cat=\'inconsistent\', rules=[\'Q-COUNT\', \'S-COUNT\'],\\n         text=\'6人局：第1个任务（1、2号）2张失败票；第2个任务（3、4、5号）1张失败票。问：6号是好人吗？答：是。\',\\n         inst={\'rec\': rec(6, C6, [Q(1, (1, 2), 2, n=6), Q(2, (3, 4, 5), 1, n=6)])},\\n         claim=GOOD(6), expect=\'inconsistent\'),\\n    # ---------------- private knowledge ----------------\\n    dict(id=\'T26\', cat=\'perspective\', rules=[\'K-MERLIN\', \'S-DEAL\'],\\n         text=\'7人局（含莫德雷德）。你是1号梅林，开局看到3号、5号是坏人。问：坏人就是3号和5号吗？答：是。\',\\n         inst={\'rec\': rec(7, C7), \'view\': view(1, \'merlin\', (3, 5))}, claim=EVILSET(3, 5), expect=\'contradicted\'),\\n    dict(id=\'T27\', cat=\'perspective\', rules=[\'K-MERLIN\', \'S-COUNT\'],\\n         text=\'同上。问：2、4、6、7号中恰好还有一名坏人吗？答：是（莫德雷德对梅林隐藏）。\',\\n         inst={\'rec\': rec(7, C7), \'view\': view(1, \'merlin\', (3, 5))}, claim=EXACTLY((2, 4, 6, 7), 1), expect=\'entailed\'),\\n    dict(id=\'T28\', cat=\'combination\', rules=[\'K-MERLIN\', \'Q-COUNT\', \'S-COUNT\'],\\n         text=\'同上，另外第1个任务由2、4号执行，翻开1张失败票。问：6号和7号都是好人吗？答：是。\',\\n         inst={\'rec\': rec(7, C7, [Q(1, (2, 4), 1, n=7)]), \'view\': view(1, \'merlin\', (3, 5))},\\n         claim=AND(GOOD(6), GOOD(7)), expect=\'entailed\'),\\n    dict(id=\'T29\', cat=\'perspective\', rules=[\'K-PERCIVAL\'],\\n         text=\'7人局。你是2号派西维尔，开局看到4号和6号（梅林与莫甘娜）。问：4号是梅林吗？答：是。\',\\n         inst={\'rec\': rec(7, C7), \'view\': view(2, \'percival\', (4, 6))}, claim=ROLE(4, \'merlin\'), expect=\'undetermined\'),\\n    dict(id=\'T30\', cat=\'combination\', rules=[\'K-PERCIVAL\', \'Q-COUNT\'],\\n         text=\'同上，另外第1个任务由4、7号执行，翻开2张失败票。问：6号是梅林吗？答：是。\',\\n         inst={\'rec\': rec(7, C7, [Q(1, (4, 7), 2, n=7)]), \'view\': view(2, \'percival\', (4, 6))},\\n         claim=ROLE(6, \'merlin\'), expect=\'entailed\'),\\n    dict(id=\'T31\', cat=\'perspective\', rules=[\'K-EVIL\', \'S-COUNT\'],\\n         text=\'7人局（含奥伯伦）。你是3号爪牙，开局只看到5号一名同伴。问：坏人只有你和5号吗？答：是。\',\\n         inst={\'rec\': rec(7, C7O), \'view\': view(3, \'minion\', (5,))}, claim=EVILSET(3, 5), expect=\'contradicted\'),\\n    dict(id=\'T32\', cat=\'perspective\', rules=[\'K-EVIL\'],\\n         text=\'同上。问：1号是好人吗？答：是。\',\\n         inst={\'rec\': rec(7, C7O), \'view\': view(3, \'minion\', (5,))}, claim=GOOD(1), expect=\'undetermined\'),\\n    dict(id=\'T33\', cat=\'combination\', rules=[\'V-SELF\', \'Q-COUNT\', \'S-COUNT\'],\\n         text=\'5人局。你是1号忠臣。第1个任务由2、3号执行，翻开2张失败票。问：4号和5号都是好人吗？答：是。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (2, 3), 2, n=5)]), \'view\': view(1, \'servant\')},\\n         claim=AND(GOOD(4), GOOD(5)), expect=\'entailed\'),\\n    dict(id=\'T34\', cat=\'inconsistent\', rules=[\'K-PERCIVAL\'],\\n         text=\'7人局。你是2号派西维尔，开局看到了3号、4号、6号三个人。问：3号是梅林吗？答：是。\',\\n         inst={\'rec\': rec(7, C7), \'view\': view(2, \'percival\', (3, 4, 6))}, claim=ROLE(3, \'merlin\'),\\n         expect=\'inconsistent\'),\\n    dict(id=\'T35\', cat=\'inconsistent\', rules=[\'V-SELF\', \'Q-GOOD\', \'Q-COUNT\'],\\n         text=\'5人局。你是1号忠臣，和2号一起执行第1个任务，翻开2张失败票。问：2号是坏人吗？答：是。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2), 2, n=5)]), \'view\': view(1, \'servant\')},\\n         claim=EVIL(2), expect=\'inconsistent\'),\\n    # ---------------- end of game ----------------\\n    dict(id=\'T36\', cat=\'simple\', rules=[\'G-MERLIN-HIT\'],\\n         text=\'5人局，前三个任务成功；刺客刺杀4号后好人获胜。问：4号不是梅林吗？答：不是梅林。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2), 0, n=5), Q(2, (1, 2, 3), 0, n=5), Q(3, (1, 3), 0, n=5)],\\n                          end={\'assassinated\': 3, \'outcome\': \'good_win\'})},\\n         claim=NOT(ROLE(4, \'merlin\')), expect=\'entailed\'),\\n    dict(id=\'T37\', cat=\'simple\', rules=[\'G-MERLIN-HIT\'],\\n         text=\'同样的任务结果，但刺杀4号后坏人获胜。问：4号是梅林吗？答：是。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2), 0, n=5), Q(2, (1, 2, 3), 0, n=5), Q(3, (1, 3), 0, n=5)],\\n                          end={\'assassinated\': 3, \'outcome\': \'evil_win\'})},\\n         claim=ROLE(4, \'merlin\'), expect=\'entailed\'),\\n    dict(id=\'T38\', cat=\'inconsistent\', rules=[\'G-ASSASSIN\'],\\n         text=\'5人局只成功了两个任务，刺客就开始刺杀。问：4号是梅林吗？答：否。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2), 0, n=5), Q(2, (1, 2, 3), 0, n=5)],\\n                          end={\'assassinated\': 3, \'outcome\': \'good_win\'})},\\n         claim=NOT(ROLE(4, \'merlin\')), expect=\'inconsistent\'),\\n    dict(id=\'T39\', cat=\'inconsistent\', rules=[\'G-END\'],\\n         text=\'5人局前三个任务都失败后，记录里还有第4个任务。问：1号是坏人吗？答：是。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2), 1, n=5), Q(2, (1, 2, 3), 1, n=5), Q(3, (1, 3), 1, n=5),\\n                                   Q(4, (1, 2, 3), 0, n=5)])},\\n         claim=EVIL(1), expect=\'inconsistent\'),\\n    # ---------------- Lady of the Lake ----------------\\n    dict(id=\'T40\', cat=\'perspective\', rules=[\'K-LADY\'],\\n         text=\'8人局。你是1号忠臣，用湖中仙女查验4号，结果为好人。问：4号是好人吗？答：是。\',\\n         inst={\'rec\': rec(8, C8), \'view\': view(1, \'servant\', (), lady=((4, False),))}, claim=GOOD(4),\\n         expect=\'entailed\'),\\n    dict(id=\'T41\', cat=\'inconsistent\', rules=[\'L-PREV\', \'L-CHAIN\'],\\n         text=\'8人局首位队长1号，湖中仙女在8号手中。第2轮后8号查验3号；第3轮后3号又查验8号。问：3号是好人吗？答：是。\',\\n         inst={\'rec\': rec(8, C8, [Q(1, (1, 2, 3), 0, leader=1, n=8), Q(2, (1, 2, 3, 4), 0, leader=2, n=8),\\n                                   Q(3, (1, 2, 3, 5), 0, leader=3, n=8)],\\n                          lady=[(2, 8, 3), (3, 3, 8)])},\\n         claim=GOOD(3), expect=\'inconsistent\'),\\n    # ---------------- encoding variants ----------------\\n    dict(id=\'T42\', cat=\'variant\', rules=[\'Q-COUNT\', \'V-FAILCOUNT\'],\\n         text=\'（只公开任务结果、不公开失败票数的读法）5人局第1个任务由1、2号执行并失败。问：1号和2号都是坏人吗？答：是。\',\\n         inst={\'rec\': rec(5, C5, [Q(1, (1, 2), 2, n=5)]), \'variant\': {\'fail_count_public\': False}},\\n         claim=AND(EVIL(1), EVIL(2)), expect=\'undetermined\'),\\n    dict(id=\'T43\', cat=\'variant\', rules=[\'K-MERLIN\', \'V-OBERON\'],\\n         text=\'10人局（含莫德雷德、奥伯伦）。你是1号梅林，开局看到3、5、7号三名坏人。问：2、4、6、8、9、10号中恰好还有一名坏人吗？答：是。\',\\n         inst={\'rec\': rec(10, C10), \'view\': view(1, \'merlin\', (3, 5, 7))},\\n         claim=EXACTLY((2, 4, 6, 8, 9, 10), 1), expect=\'entailed\'),\\n    dict(id=\'T44\', cat=\'variant\', rules=[\'K-MERLIN\', \'V-OBERON\'],\\n         text=\'同上，但按“奥伯伦不向梅林显示”的读法。答：是。\',\\n         inst={\'rec\': rec(10, C10), \'view\': view(1, \'merlin\', (3, 5, 7)), \'variant\': {\'merlin_sees_oberon\': False}},\\n         claim=EXACTLY((2, 4, 6, 8, 9, 10), 1), expect=\'inconsistent\'),\\n]\\n", "avalon/selfcheck.py": "\\"\\"\\"Acceptance checks.  Runs everything that does not need Z3 with the oracle\\nalone; when z3 is importable it also cross-checks the SMT encoding.\\n\\n  python selfcheck.py [--quick] [--out ../results/selfcheck]\\n\\"\\"\\"\\nimport argparse\\nimport json\\nimport random\\nimport sys\\nimport time\\nfrom pathlib import Path\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parent))\\nimport game\\nimport oracle as O\\nfrom cases import CASES\\nfrom search import min_reveal_naive, min_reveal_cegar, adaptive_depth, candidates\\nfrom sim import play, true_view, prefix, random_claim, corrupt\\n\\ntry:\\n    from smt import SMT\\n    HAVE_Z3 = True\\nexcept ImportError:\\n    HAVE_Z3 = False\\n\\n\\ndef tables_agree():\\n    ok = game.EVIL_COUNT == O.O_EVIL_COUNT\\n    ok &= all(list(game.TEAM_SIZE[n]) == O.O_SIZES[n] for n in game.TEAM_SIZE)\\n    ok &= all(game.fails_required(n, q) == O.o_need(n, q) for n in range(5, 11) for q in range(1, 6))\\n    ok &= set(game.EVIL_ROLES) == O.O_EVIL\\n    return bool(ok)\\n\\n\\ndef run_cases(backends):\\n    rows = []\\n    for c in CASES:\\n        row = {\'id\': c[\'id\'], \'cat\': c[\'cat\'], \'expect\': c[\'expect\'], \'rules\': c[\'rules\']}\\n        for name, b in backends.items():\\n            res = b.classify(c[\'inst\'], c[\'claim\'], explain=True) if name == \'z3\' else b.classify(c[\'inst\'], c[\'claim\'])\\n            row[name] = res[\'label\']\\n            if \'core\' in res:\\n                row[\'core\'] = res[\'core\']\\n            if res.get(\'positive\'):\\n                row[name + \'_pos\'] = res[\'positive\']\\n            if res.get(\'negative\'):\\n                row[name + \'_neg\'] = res[\'negative\']\\n        row[\'pass\'] = all(row[name] == c[\'expect\'] for name in backends)\\n        rows.append(row)\\n    return rows\\n\\n\\ndef sample_instances(rng, n_games, max_n=8):\\n    insts = []\\n    for g in range(n_games):\\n        n = rng.choice([5, 5, 6, 6, 7, 7, 8] + ([9, 10] if max_n >= 10 else []))\\n        rec, roles = play(rng, n=n, p_approve=0.6, lady=(n >= 7 and rng.random() < 0.5))\\n        done = len(rec[\'quests\'])\\n        for k in sorted({rng.randint(1, done), done}):\\n            r = prefix(rec, k) if k < done else rec\\n            for persp in (None, rng.randrange(n)):\\n                view = None if persp is None else true_view(roles, persp, rec=r)\\n                variant = rng.choice([{}, {}, {\'fail_count_public\': False}, {\'merlin_sees_oberon\': False}])\\n                if variant.get(\'merlin_sees_oberon\') is False and view is not None:\\n                    view = true_view(roles, persp, variant, rec=r)\\n                insts.append(({\'rec\': r, \'view\': view, \'variant\': variant}, roles, f\'g{g}k{k}p{persp}\'))\\n    return insts\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\'--quick\', action=\'store_true\')\\n    ap.add_argument(\'--seed\', type=int, default=20261112)\\n    ap.add_argument(\'--out\', default=str(Path(__file__).resolve().parents[1] / \'results\' / \'selfcheck\'))\\n    args = ap.parse_args()\\n    out = Path(args.out)\\n    out.mkdir(parents=True, exist_ok=True)\\n    rng = random.Random(args.seed)\\n    t0 = time.perf_counter()\\n    oracle = O.Oracle()\\n    backends = {\'oracle\': oracle}\\n    if HAVE_Z3:\\n        backends[\'z3\'] = SMT()\\n    summary = {\'z3_available\': HAVE_Z3, \'seed\': args.seed, \'tables_agree\': tables_agree()}\\n\\n    # 1. hand-written cases\\n    rows = run_cases(backends)\\n    summary[\'cases\'] = {\'n\': len(rows), \'pass\': sum(r[\'pass\'] for r in rows),\\n                        \'by_expect\': {k: sum(r[\'expect\'] == k for r in rows)\\n                                      for k in (\'entailed\', \'contradicted\', \'undetermined\', \'inconsistent\')},\\n                        \'failed\': [r[\'id\'] for r in rows if not r[\'pass\']]}\\n    (out / \'cases.json\').write_text(json.dumps(rows, ensure_ascii=False, indent=1), encoding=\'utf-8\')\\n\\n    # 2. random instances: soundness w.r.t. the true roles, and oracle vs z3\\n    insts = sample_instances(rng, 25 if args.quick else 120, max_n=8 if args.quick else 10)\\n    agree = sound = total = 0\\n    mismatches = []\\n    for inst, roles, tag in insts:\\n        for _ in range(3):\\n            claim = random_claim(rng, inst[\'rec\'][\'n\'])\\n            a = oracle.classify(inst, claim)[\'label\']\\n            truth = O.claim_holds(claim, roles, inst[\'rec\'])\\n            total += 1\\n            sound += (a == \'entailed\' and truth) or (a == \'contradicted\' and not truth) or a == \'undetermined\'\\n            if HAVE_Z3:\\n                b = backends[\'z3\'].classify(inst, claim)[\'label\']\\n                agree += a == b\\n                if a != b:\\n                    mismatches.append({\'tag\': tag, \'claim\': claim, \'oracle\': a, \'z3\': b, \'inst\': inst})\\n    summary[\'random\'] = {\'instances\': len(insts), \'claims\': total, \'sound_wrt_truth\': sound,\\n                         \'oracle_z3_agree\': agree if HAVE_Z3 else None}\\n\\n    # 3. corrupted records must be inconsistent for both backends\\n    bad_ok = bad_total = 0\\n    for inst, roles, tag in insts[:60]:\\n        r, kind = corrupt(rng, inst[\'rec\'])\\n        ci = {**inst, \'rec\': r, \'variant\': {}}  # violations of the fail count are visible only in the default reading\\n        labels = {name: b.classify(ci, {\'t\': \'evil\', \'p\': 0})[\'label\'] for name, b in backends.items()}\\n        bad_total += 1\\n        bad_ok += all(v == \'inconsistent\' for v in labels.values())\\n        if not all(v == \'inconsistent\' for v in labels.values()):\\n            mismatches.append({\'tag\': tag + \':corrupt:\' + kind, \'labels\': labels, \'inst\': ci})\\n    summary[\'corrupted\'] = {\'n\': bad_total, \'all_inconsistent\': bad_ok}\\n\\n    # 4. minimum reveal sets: brute force vs naive vs CEGAR (oracle backend), and z3 CEGAR\\n    mr = {\'n\': 0, \'naive_eq\': 0, \'cegar_eq\': 0, \'z3_cegar_eq\': 0 if HAVE_Z3 else None, \'adaptive_le_oneshot\': 0}\\n    for inst, roles, tag in insts[: (15 if args.quick else 80)]:\\n        if inst[\'rec\'][\'n\'] > 8:\\n            continue\\n        claim = random_claim(rng, inst[\'rec\'][\'n\'])\\n        if oracle.classify(inst, claim)[\'label\'] != \'undetermined\':\\n            continue\\n        cands = candidates(inst)\\n        brute = oracle.min_reveal_sets(inst, claim, cands)\\n        size = len(brute[0]) if brute else None  # [] = no sufficient set exists\\n        nv = min_reveal_naive(oracle, inst, claim, cands)\\n        cg = min_reveal_cegar(oracle, inst, claim, cands)\\n        mr[\'n\'] += 1\\n        mr[\'naive_eq\'] += (nv.get(\'size\') == size and sorted(map(sorted, nv.get(\'sets\', []))) == sorted(map(sorted, brute or [])))\\n        mr[\'cegar_eq\'] += cg.get(\'size\') == size and (cg.get(\'set\') is None or sorted(cg[\'set\']) in [sorted(s) for s in brute])\\n        if HAVE_Z3:\\n            zc = min_reveal_cegar(backends[\'z3\'], inst, claim, cands)\\n            mr[\'z3_cegar_eq\'] += zc.get(\'size\') == size\\n        d, _ = adaptive_depth(oracle, inst, claim, cands)\\n        # no reveal strategy exists  <=>  no one-shot set exists; otherwise adaptive <= one-shot\\n        mr[\'adaptive_le_oneshot\'] += (d is None and size is None) or (d is not None and size is not None and d <= size)\\n        mr.setdefault(\'adaptive_lt_oneshot\', 0)\\n        mr[\'adaptive_lt_oneshot\'] += d is not None and size is not None and d < size\\n        mr.setdefault(\'no_reveal_set\', 0)\\n        mr[\'no_reveal_set\'] += size is None\\n    summary[\'min_reveal\'] = mr\\n\\n    # 5. votes carry no logical information about roles (rule-level property)\\n    vp = {\'n\': 0, \'same\': 0}\\n    for inst, roles, tag in insts[:40]:\\n        stripped = {**inst, \'rec\': {**inst[\'rec\'], \'quests\': [\\n            {**qd, \'proposals\': [{**p, \'approve\': None} for p in qd.get(\'proposals\', [])]}\\n            for qd in inst[\'rec\'][\'quests\']]}}\\n        for p in range(inst[\'rec\'][\'n\']):\\n            c = {\'t\': \'evil\', \'p\': p}\\n            vp[\'n\'] += 1\\n            vp[\'same\'] += oracle.classify(inst, c)[\'label\'] == oracle.classify(stripped, c)[\'label\']\\n    summary[\'votes_uninformative\'] = vp\\n\\n    summary[\'seconds\'] = round(time.perf_counter() - t0, 2)\\n    (out / \'summary.json\').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding=\'utf-8\')\\n    (out / \'mismatches.json\').write_text(json.dumps(mismatches[:50], ensure_ascii=False, indent=1), encoding=\'utf-8\')\\n    print(json.dumps(summary, ensure_ascii=False, indent=2))\\n    ok = (summary[\'tables_agree\'] and not summary[\'cases\'][\'failed\'] and sound == total\\n          and bad_ok == bad_total and (not HAVE_Z3 or agree == total)\\n          and mr[\'naive_eq\'] == mr[\'n\'] and mr[\'cegar_eq\'] == mr[\'n\'] and mr[\'adaptive_le_oneshot\'] == mr[\'n\']\\n          and (not HAVE_Z3 or mr[\'z3_cegar_eq\'] == mr[\'n\']) and vp[\'same\'] == vp[\'n\'])\\n    print(\'ALL CHECKS PASSED\' if ok else \'SOME CHECKS FAILED\')\\n    return 0 if ok else 1\\n\\n\\nif __name__ == \'__main__\':\\n    sys.exit(main())\\n", "avalon/logs.py": "\\"\\"\\"Parse avalonlogs (github.com/WhoaWhoa/avalonlogs) into records, and audit\\nthe real games against the formal rules.\\n\\n  python logs.py --data ../data/avalonlogs --inspect     # schema statistics\\n  python logs.py --data ../data/avalonlogs --audit       # parse + rule audit\\n\\nObserved schema (one JSON file per game):\\n  players  : [{name, uid}]                       seat order (verified by the audit)\\n  missions : [{teamSize, failsRequired, numFails, state, team: [name],\\n               proposals: [{proposer, team: [name], votes: [name], state}]}]\\n  outcome  : {state, message, assassinated, roles: [{name, role, assassin}], ...}\\nTwo things are not documented and are decided from the data by the audit:\\nwhether `votes` lists approvers or rejecters, and the direction in which\\nleadership passes along the `players` array.\\n\\"\\"\\"\\nimport argparse\\nimport json\\nimport sys\\nfrom collections import Counter\\nfrom pathlib import Path\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parent))\\nfrom game import normalize_role, EVIL_ROLES, TEAM_SIZE, fails_required, config_ok\\nimport oracle as O\\n\\nROOT = Path(__file__).resolve().parents[1]\\n\\n\\ndef game_files(data_dir):\\n    d = Path(data_dir)\\n    base = d / \'logs\' if (d / \'logs\').is_dir() else d\\n    return sorted(p for p in base.iterdir() if p.is_file() and not p.name.startswith(\'.\')\\n                  and p.suffix.lower() not in (\'.md\', \'.py\', \'.txt\', \'.ipynb\'))\\n\\n\\ndef load(path):\\n    with open(path, encoding=\'utf-8\') as f:\\n        return json.load(f)\\n\\n\\ndef _state(s):\\n    s = str(s or \'\').upper()\\n    if \'SUCC\' in s:\\n        return \'success\'\\n    if \'FAIL\' in s:\\n        return \'fail\'\\n    if \'APPROV\' in s:\\n        return \'approved\'\\n    if \'REJECT\' in s:\\n        return \'rejected\'\\n    return None\\n\\n\\ndef inspect(files, limit=None):\\n    keys, mkeys, pkeys, okeys = Counter(), Counter(), Counter(), Counter()\\n    mstates, pstates, ostates, roles, ns = Counter(), Counter(), Counter(), Counter(), Counter()\\n    bad = 0\\n    for path in files[:limit]:\\n        try:\\n            g = load(path)\\n        except Exception:\\n            bad += 1\\n            continue\\n        keys.update(g.keys())\\n        ns[len(g.get(\'players\', []))] += 1\\n        for m in g.get(\'missions\', []):\\n            mkeys.update(m.keys())\\n            mstates[str(m.get(\'state\'))] += 1\\n            for p in m.get(\'proposals\', []):\\n                pkeys.update(p.keys())\\n                pstates[str(p.get(\'state\'))] += 1\\n        o = g.get(\'outcome\') or {}\\n        okeys.update(o.keys())\\n        ostates[str(o.get(\'state\'))] += 1\\n        for r in o.get(\'roles\', []) or []:\\n            roles[str(r.get(\'role\'))] += 1\\n    return {\'files\': len(files[:limit]), \'unreadable\': bad, \'top_keys\': dict(keys), \'mission_keys\': dict(mkeys),\\n            \'proposal_keys\': dict(pkeys), \'outcome_keys\': dict(okeys), \'mission_states\': dict(mstates),\\n            \'proposal_states\': dict(pstates), \'outcome_states\': dict(ostates), \'roles\': dict(roles),\\n            \'players\': dict(ns)}\\n\\n\\ndef parse(g, votes_are=\'approve\', reverse_seats=False, game_id=None):\\n    \\"\\"\\"Return (record, true_roles, issues) or (None, None, issues).\\"\\"\\"\\n    issues = []\\n    names = [p[\'name\'] for p in g.get(\'players\', [])]\\n    n = len(names)\\n    if not 5 <= n <= 10 or len(set(names)) != n:\\n        return None, None, [\'players\']\\n    seat = {name: (-i) % n if reverse_seats else i for i, name in enumerate(names)}\\n    o = g.get(\'outcome\') or {}\\n    try:\\n        rmap = {r[\'name\']: normalize_role(r[\'role\']) for r in o.get(\'roles\') or []}\\n    except ValueError as e:\\n        return None, None, [\'role:\' + str(e)]\\n    if set(rmap) != set(names):\\n        return None, None, [\'roles_missing\']\\n    roles = [None] * n\\n    for name, r in rmap.items():\\n        roles[seat[name]] = r\\n    config = dict(Counter(roles))\\n    quests = []\\n    for k, m in enumerate(g.get(\'missions\') or []):\\n        q = k + 1\\n        res = _state(m.get(\'state\'))\\n        res = res if res in (\'success\', \'fail\') else None\\n        props = []\\n        for p in m.get(\'proposals\') or []:\\n            st = _state(p.get(\'state\'))\\n            if st not in (\'approved\', \'rejected\'):\\n                continue\\n            try:\\n                team = sorted(seat[x] for x in p.get(\'team\') or [])\\n                voters = {seat[x] for x in p.get(\'votes\') or []}\\n            except KeyError:\\n                return None, None, [\'unknown_name\']\\n            approve = sorted(voters) if votes_are == \'approve\' else sorted(set(range(n)) - voters)\\n            props.append({\'leader\': seat.get(p.get(\'proposer\'), -1), \'team\': team,\\n                          \'approve\': approve, \'approved\': st == \'approved\'})\\n        if res is None:\\n            if len(props) == 5 and not props[-1][\'approved\']:\\n                quests.append({\'q\': q, \'proposals\': props, \'team\': [], \'fails\': None, \'result\': None})\\n            break\\n        try:\\n            team = sorted(seat[x] for x in m.get(\'team\') or [])\\n        except KeyError:\\n            return None, None, [\'unknown_name\']\\n        if m.get(\'failsRequired\') is not None and m[\'failsRequired\'] != fails_required(n, q):\\n            issues.append(\'platform_failsRequired\')\\n        if m.get(\'teamSize\') is not None and m[\'teamSize\'] != TEAM_SIZE[n][q - 1]:\\n            issues.append(\'platform_teamSize\')\\n        quests.append({\'q\': q, \'proposals\': props, \'team\': team,\\n                       \'fails\': m.get(\'numFails\'), \'result\': res})\\n    st = str(o.get(\'state\') or \'\').upper()\\n    outcome = \'good_win\' if \'GOOD\' in st else (\'evil_win\' if \'EVIL\' in st else None)\\n    a = o.get(\'assassinated\')\\n    end = {\'assassinated\': seat.get(a) if a else None, \'outcome\': outcome}\\n    assassin = [seat[r[\'name\']] for r in o.get(\'roles\') or [] if r.get(\'assassin\')]\\n    rec = {\'n\': n, \'config\': config, \'quests\': quests, \'end\': end if outcome else None,\\n           \'source\': {\'dataset\': \'avalonlogs\', \'game\': game_id,\\n                      \'assassin\': assassin[0] if assassin else None}}\\n    return rec, tuple(roles), issues\\n\\n\\ndef vote_semantics(games):\\n    \\"\\"\\"Fraction of decided proposals consistent with \'votes = approvers\' vs \'votes = rejecters\'.\\"\\"\\"\\n    a = b = tot = 0\\n    for g in games:\\n        n = len(g.get(\'players\', []))\\n        for m in g.get(\'missions\') or []:\\n            for p in m.get(\'proposals\') or []:\\n                st = _state(p.get(\'state\'))\\n                if st not in (\'approved\', \'rejected\'):\\n                    continue\\n                k = len(set(p.get(\'votes\') or []))\\n                tot += 1\\n                a += (2 * k > n) == (st == \'approved\')\\n                b += (2 * (n - k) > n) == (st == \'approved\')\\n    return {\'proposals\': tot, \'votes_are_approvers\': a, \'votes_are_rejecters\': b}\\n\\n\\ndef leader_direction(games):\\n    plus = minus = other = 0\\n    for g in games:\\n        names = [p[\'name\'] for p in g.get(\'players\', [])]\\n        n = len(names)\\n        idx = {x: i for i, x in enumerate(names)}\\n        seq = [idx.get(p.get(\'proposer\')) for m in g.get(\'missions\') or [] for p in m.get(\'proposals\') or []\\n               if _state(p.get(\'state\')) in (\'approved\', \'rejected\')]\\n        for x, y in zip(seq, seq[1:]):\\n            if x is None or y is None:\\n                other += 1\\n            elif y == (x + 1) % n:\\n                plus += 1\\n            elif y == (x - 1) % n:\\n                minus += 1\\n            else:\\n                other += 1\\n    return {\'next_in_array\': plus, \'previous_in_array\': minus, \'other\': other}\\n\\n\\ndef audit(data_dir, out_dir, limit=None):\\n    files = game_files(data_dir)[:limit]\\n    games, ids = [], []\\n    for f in files:\\n        try:\\n            games.append(load(f))\\n            ids.append(f.name)\\n        except Exception:\\n            pass\\n    vs = vote_semantics(games)\\n    ld = leader_direction(games)\\n    votes_are = \'approve\' if vs[\'votes_are_approvers\'] >= vs[\'votes_are_rejecters\'] else \'reject\'\\n    reverse = ld[\'previous_in_array\'] > ld[\'next_in_array\']\\n    parsed, skipped, viol, truth_bad, platform = [], Counter(), Counter(), Counter(), Counter()\\n    per_game = []\\n    for g, gid in zip(games, ids):\\n        rec, roles, issues = parse(g, votes_are, reverse, gid)\\n        platform.update(i for i in issues if i.startswith(\'platform\'))\\n        if rec is None:\\n            skipped[issues[0]] += 1\\n            continue\\n        if config_ok(rec[\'n\'], rec[\'config\']):\\n            skipped[\'config:\' + \',\'.join(config_ok(rec[\'n\'], rec[\'config\']))] += 1\\n            continue\\n        v = O.record_violations(rec, {})\\n        viol.update(v)\\n        truth = O.world_ok(roles, rec, None, [], {}) if not v else None\\n        if truth is False:\\n            truth_bad[\'public_record\'] += 1\\n        for p in range(rec[\'n\']):\\n            from sim import true_view\\n            if not v and not O.world_ok(roles, rec, true_view(roles, p), [], {}):\\n                truth_bad[\'view_\' + roles[p]] += 1\\n        per_game.append({\'game\': gid, \'n\': rec[\'n\'], \'violations\': v, \'truth_consistent\': truth})\\n        parsed.append({\'rec\': rec, \'roles\': list(roles)})\\n    out = Path(out_dir)\\n    out.mkdir(parents=True, exist_ok=True)\\n    summary = {\\n        \'files\': len(files), \'loaded\': len(games), \'parsed\': len(parsed), \'skipped\': dict(skipped),\\n        \'vote_semantics\': vs, \'votes_interpreted_as\': votes_are,\\n        \'leader_direction\': ld, \'seats_reversed\': reverse,\\n        \'platform_table_mismatch\': dict(platform),\\n        \'games_with_rule_violations\': sum(1 for x in per_game if x[\'violations\']),\\n        \'violations_by_rule\': dict(viol),\\n        \'true_roles_inconsistent\': dict(truth_bad),\\n        \'players\': dict(Counter(x[\'n\'] for x in per_game)),\\n        \'configs\': dict(Counter(json.dumps(p[\'rec\'][\'config\'], sort_keys=True) for p in parsed).most_common(15)),\\n    }\\n    (out / \'audit_summary.json\').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding=\'utf-8\')\\n    (out / \'audit_games.jsonl\').write_text(\'\\\\n\'.join(json.dumps(x) for x in per_game), encoding=\'utf-8\')\\n    with open(out / \'parsed_games.jsonl\', \'w\', encoding=\'utf-8\') as f:\\n        for p in parsed:\\n            f.write(json.dumps(p, ensure_ascii=False) + \'\\\\n\')\\n    return summary\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\'--data\', default=str(ROOT / \'data\' / \'avalonlogs\'))\\n    ap.add_argument(\'--out\', default=str(ROOT / \'results\' / \'audit\'))\\n    ap.add_argument(\'--inspect\', action=\'store_true\')\\n    ap.add_argument(\'--audit\', action=\'store_true\')\\n    ap.add_argument(\'--limit\', type=int)\\n    a = ap.parse_args()\\n    files = game_files(a.data)\\n    if a.inspect:\\n        print(json.dumps(inspect(files, a.limit), ensure_ascii=False, indent=2))\\n    if a.audit:\\n        print(json.dumps(audit(a.data, a.out, a.limit), ensure_ascii=False, indent=2))\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "avalon/analyze.py": "\\"\\"\\"What do the rules alone let players deduce in real games?\\n\\nInput : results/audit/parsed_games.jsonl (from logs.py --audit)\\nOutput: results/analysis/*.json\\n\\nA1 deducibility : after each quest, for the public observer and for every\\n                  player\'s true private view, how many other players\'\\n                  alignments are logically fixed by the rules?\\nA2 variant      : the same when only quest results (not Fail counts) are public\\nA3 assassination: from the assassin\'s view before the shot, how many players\\n                  can still be Merlin, and was the shot among them?\\nA4 completion   : for the public observer at the end of the game, the minimum\\n                  number of loyalty reveals (Lady of the Lake checks) that\\n                  decide the full evil team: one-shot (CEGAR, naive) vs\\n                  adaptive, with solver calls and run time.\\n\\n  python analyze.py --games 600 [--backend z3|oracle]\\n\\"\\"\\"\\nimport argparse\\nimport json\\nimport random\\nimport sys\\nimport time\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parent))\\nimport oracle as O\\nfrom game import EVIL_ROLES\\nfrom sim import true_view, prefix\\nfrom search import min_reveal_cegar, min_reveal_naive, adaptive_depth, candidates\\n\\nROOT = Path(__file__).resolve().parents[1]\\n\\n\\ndef get_backend(name):\\n    if name in (\'z3\', \'auto\'):\\n        try:\\n            from smt import SMT\\n            return SMT()\\n        except ImportError:\\n            if name == \'z3\':\\n                raise\\n    return O.Oracle()\\n\\n\\ndef determined(backend, inst):\\n    if hasattr(backend, \'determined_players\'):\\n        return backend.determined_players(inst)\\n    return O.determined_players(inst)\\n\\n\\ndef possible_roles(backend, inst, p):\\n    if hasattr(backend, \'possible_roles\'):\\n        return backend.possible_roles(inst, p)\\n    return O.possible_roles(inst, p)\\n\\n\\ndef mean(xs):\\n    return sum(xs) / len(xs) if xs else None\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\'--parsed\', default=str(ROOT / \'results\' / \'audit\' / \'parsed_games.jsonl\'))\\n    ap.add_argument(\'--out\', default=str(ROOT / \'results\' / \'analysis\'))\\n    ap.add_argument(\'--games\', type=int, default=600)\\n    ap.add_argument(\'--completion-games\', type=int, default=150)\\n    ap.add_argument(\'--backend\', default=\'auto\')\\n    ap.add_argument(\'--seed\', type=int, default=271)\\n    a = ap.parse_args()\\n    backend = get_backend(a.backend)\\n    rng = random.Random(a.seed)\\n    games = [json.loads(x) for x in open(a.parsed, encoding=\'utf-8\') if x.strip()]\\n    games = [g for g in games if not O.record_violations(g[\'rec\'], {})]\\n    if backend.name == \'oracle\':\\n        games = [g for g in games if g[\'rec\'][\'n\'] <= 8]\\n    rng.shuffle(games)\\n    sample = games[: a.games]\\n    out = Path(a.out)\\n    out.mkdir(parents=True, exist_ok=True)\\n    t0 = time.perf_counter()\\n\\n    # ---------------- A1 / A2 ----------------\\n    acc = defaultdict(list)\\n    full = defaultdict(list)\\n    rows = []\\n    for gi, g in enumerate(sample):\\n        rec, roles = g[\'rec\'], tuple(g[\'roles\'])\\n        n = rec[\'n\']\\n        done = [k for k, qd in enumerate(rec[\'quests\'], 1) if qd.get(\'result\') is not None]\\n        for k in done:\\n            r = prefix(rec, k)\\n            for persp in [None] + list(range(n)):\\n                view = None if persp is None else true_view(roles, persp)\\n                who = \'observer\' if persp is None else roles[persp]\\n                for variant_name, variant in ((\'fails_public\', {}), (\'result_only\', {\'fail_count_public\': False})):\\n                    det = determined(backend, {\'rec\': r, \'view\': view, \'variant\': variant})\\n                    if det is None:\\n                        continue\\n                    others = [j for j in range(n) if j != persp]\\n                    fixed = sum(det[j] != \'unknown\' for j in others)\\n                    key = (variant_name, who, k)\\n                    acc[key].append(fixed / len(others))\\n                    full[key].append(all(det[j] != \'unknown\' for j in others))\\n                    if variant_name == \'fails_public\':\\n                        rows.append({\'game\': rec[\'source\'][\'game\'], \'n\': n, \'quest\': k, \'perspective\': who,\\n                                     \'fixed\': fixed, \'others\': len(others)})\\n        if gi % 50 == 0:\\n            print(f\'A1 {gi}/{len(sample)} {time.perf_counter() - t0:.0f}s\', flush=True)\\n    a1 = [{\'variant\': v, \'perspective\': w, \'after_quest\': k, \'states\': len(xs),\\n           \'mean_fraction_fixed\': round(mean(xs), 4), \'all_fixed_rate\': round(mean(full[(v, w, k)]), 4)}\\n          for (v, w, k), xs in sorted(acc.items())]\\n    (out / \'A1_A2_deducibility.json\').write_text(json.dumps(a1, ensure_ascii=False, indent=1), encoding=\'utf-8\')\\n    with open(out / \'A1_states.jsonl\', \'w\', encoding=\'utf-8\') as f:\\n        for r in rows:\\n            f.write(json.dumps(r) + \'\\\\n\')\\n\\n    # ---------------- A3 ----------------\\n    a3 = []\\n    for g in sample:\\n        rec, roles = g[\'rec\'], tuple(g[\'roles\'])\\n        end = rec.get(\'end\') or {}\\n        assassin = rec[\'source\'].get(\'assassin\')\\n        if end.get(\'assassinated\') is None or assassin is None:\\n            continue\\n        r = {**rec, \'end\': None}\\n        view = true_view(roles, assassin)\\n        cands = [p for p in range(rec[\'n\']) if \'merlin\' in possible_roles(backend, {\'rec\': r, \'view\': view}, p)]\\n        a3.append({\'game\': rec[\'source\'][\'game\'], \'n\': rec[\'n\'], \'merlin_candidates\': len(cands),\\n                   \'shot_in_candidates\': end[\'assassinated\'] in cands,\\n                   \'hit\': roles[end[\'assassinated\']] == \'merlin\'})\\n    a3s = {\'games\': len(a3), \'mean_candidates\': mean([x[\'merlin_candidates\'] for x in a3]),\\n           \'unique_candidate_rate\': mean([x[\'merlin_candidates\'] == 1 for x in a3]),\\n           \'shot_outside_candidates\': sum(not x[\'shot_in_candidates\'] for x in a3),\\n           \'hit_rate\': mean([x[\'hit\'] for x in a3]),\\n           \'by_n\': {n: {\'games\': len(v), \'mean_candidates\': mean([x[\'merlin_candidates\'] for x in v]),\\n                        \'hit_rate\': mean([x[\'hit\'] for x in v])}\\n                    for n, v in sorted(group(a3, \'n\').items())}}\\n    (out / \'A3_assassination.json\').write_text(json.dumps({\'summary\': a3s, \'games\': a3}, indent=1), encoding=\'utf-8\')\\n\\n    # ---------------- A4 ----------------\\n    a4 = []\\n    for g in sample[: a.completion_games]:\\n        rec, roles = g[\'rec\'], tuple(g[\'roles\'])\\n        r = {**rec, \'end\': None}\\n        evil = [i for i, x in enumerate(roles) if x in EVIL_ROLES]\\n        inst = {\'rec\': r}\\n        claim = {\'t\': \'evil_set\', \'ps\': evil}\\n        if backend.classify(inst, claim)[\'label\'] != \'undetermined\':\\n            continue\\n        row = {\'game\': rec[\'source\'][\'game\'], \'n\': rec[\'n\'], \'quests\': len(rec[\'quests\'])}\\n        c0 = getattr(backend, \'calls\', 0)\\n        t = time.perf_counter()\\n        cg = min_reveal_cegar(backend, inst, claim)\\n        row.update(cegar_size=cg.get(\'size\'), cegar_calls=cg[\'calls\'], cegar_s=round(time.perf_counter() - t, 3))\\n        if rec[\'n\'] <= 7:\\n            t = time.perf_counter()\\n            nv = min_reveal_naive(backend, inst, claim, deadline=60)\\n            row.update(naive_size=nv.get(\'size\'), naive_checks=nv[\'checks\'], naive_s=round(time.perf_counter() - t, 3),\\n                       naive_status=nv[\'status\'], n_optimal_sets=len(nv.get(\'sets\', [])))\\n            t = time.perf_counter()\\n            d, first = adaptive_depth(backend, inst, claim)\\n            row.update(adaptive_depth=d, adaptive_s=round(time.perf_counter() - t, 3))\\n        row[\'backend_calls\'] = getattr(backend, \'calls\', 0) - c0\\n        a4.append(row)\\n    a4s = {\'states\': len(a4),\\n           \'by_n\': {n: {\'states\': len(v), \'mean_cegar_size\': mean([x[\'cegar_size\'] for x in v if x[\'cegar_size\'] is not None]),\\n                        \'mean_cegar_s\': mean([x[\'cegar_s\'] for x in v]),\\n                        \'mean_naive_s\': mean([x[\'naive_s\'] for x in v if \'naive_s\' in x]),\\n                        \'adaptive_lt_oneshot\': sum(1 for x in v if x.get(\'adaptive_depth\') is not None\\n                                                   and x[\'cegar_size\'] is not None and x[\'adaptive_depth\'] < x[\'cegar_size\']),\\n                        \'with_adaptive\': sum(1 for x in v if x.get(\'adaptive_depth\') is not None)}\\n                    for n, v in sorted(group(a4, \'n\').items())}}\\n    (out / \'A4_completion.json\').write_text(json.dumps({\'summary\': a4s, \'states\': a4}, indent=1), encoding=\'utf-8\')\\n\\n    meta = {\'backend\': backend.name, \'games_available\': len(games), \'games_sampled\': len(sample),\\n            \'seed\': a.seed, \'seconds\': round(time.perf_counter() - t0, 1)}\\n    (out / \'meta.json\').write_text(json.dumps(meta, indent=2), encoding=\'utf-8\')\\n    print(json.dumps({\'meta\': meta, \'A3\': a3s, \'A4\': a4s}, indent=2))\\n\\n\\ndef group(rows, key):\\n    g = defaultdict(list)\\n    for r in rows:\\n        g[r[key]].append(r)\\n    return g\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "avalon/llm_tasks.py": "\\"\\"\\"Build the frozen LLM evaluation set from real (or simulated) game states.\\n\\nEach item = one game state (record prefix + perspective) + one claim, with the\\ngold verdict computed by the solver.  Inconsistent items are made by\\nintroducing one rule violation that is visible in the text.  Votes are kept in\\nthe narrative on purpose: no rule links votes to roles, so they are logically\\nirrelevant distractors.\\n\\n  python llm_tasks.py --parsed ../results/audit/parsed_games.jsonl --out ../data/llm\\n\\"\\"\\"\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport random\\nimport sys\\nfrom collections import Counter\\nfrom pathlib import Path\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parent))\\nimport oracle as O\\nfrom game import (EVIL_ROLES, EVIL_COUNT, TEAM_SIZE, ZH_ROLE, render_record, render_view,\\n                  player_label, team_label, fails_required)\\nfrom sim import true_view, prefix\\nfrom search import min_reveal_cegar, candidates\\n\\nROOT = Path(__file__).resolve().parents[1]\\nLABELS = (\'entailed\', \'contradicted\', \'undetermined\', \'inconsistent\')\\nZH_LABEL = {\'entailed\': \'成立\', \'contradicted\': \'不成立\', \'undetermined\': \'不确定\', \'inconsistent\': \'矛盾\'}\\nQUOTA = {\'entailed\': 150, \'contradicted\': 150, \'undetermined\': 150, \'inconsistent\': 60}\\n\\n\\ndef render_claim(c):\\n    t = c[\'t\']\\n    if t == \'evil\':\\n        return f\'{player_label(c[\\"p\\"])}一定是坏人。\'\\n    if t == \'good\':\\n        return f\'{player_label(c[\\"p\\"])}一定是好人。\'\\n    if t == \'role\':\\n        return f\'{player_label(c[\\"p\\"])}一定是{ZH_ROLE[c[\\"r\\"]]}。\'\\n    if t == \'not\' and c[\'a\'][\'t\'] == \'role\':\\n        return f\'{player_label(c[\\"a\\"][\\"p\\"])}一定不是{ZH_ROLE[c[\\"a\\"][\\"r\\"]]}。\'\\n    if t == \'atleast\':\\n        return f\'{team_label(c[\\"ps\\"])}中至少有{c[\\"k\\"]}名坏人。\'\\n    if t == \'exactly\':\\n        return f\'{team_label(c[\\"ps\\"])}中恰好有{c[\\"k\\"]}名坏人。\'\\n    if t == \'same\':\\n        a, b = c[\'ps\']\\n        return f\'{player_label(a)}和{player_label(b)}一定属于同一阵营。\'\\n    if t == \'evil_set\':\\n        return f\'本局的坏人恰好是{team_label(c[\\"ps\\"])}。\'\\n    raise ValueError(t)\\n\\n\\ndef claim_pool(rng, inst, roles):\\n    rec, view = inst[\'rec\'], inst.get(\'view\')\\n    n = rec[\'n\']\\n    me = view[\'player\'] if view else None\\n    others = [j for j in range(n) if j != me]\\n    pool = []\\n    for j in others:\\n        pool += [{\'t\': \'evil\', \'p\': j}, {\'t\': \'good\', \'p\': j}]\\n    for j in rng.sample(others, min(3, len(others))):\\n        pool.append({\'t\': \'role\', \'p\': j, \'r\': \'merlin\'})\\n        pool.append({\'t\': \'not\', \'a\': {\'t\': \'role\', \'p\': j, \'r\': \'merlin\'}})\\n    for qd in rec[\'quests\']:\\n        if qd.get(\'result\') is None:\\n            continue\\n        team = qd[\'team\']\\n        for k in (1, 2):\\n            if k <= len(team):\\n                pool.append({\'t\': \'atleast\', \'ps\': team, \'k\': k})\\n        pool.append({\'t\': \'exactly\', \'ps\': team, \'k\': 1})\\n    for _ in range(3):\\n        a, b = rng.sample(others, 2)\\n        pool.append({\'t\': \'same\', \'ps\': sorted([a, b])})\\n    evil = [i for i, r in enumerate(roles) if r in EVIL_ROLES]\\n    pool.append({\'t\': \'evil_set\', \'ps\': evil})\\n    wrong = sorted(rng.sample(range(n), len(evil)))\\n    if wrong != evil:\\n        pool.append({\'t\': \'evil_set\', \'ps\': wrong})\\n    rng.shuffle(pool)\\n    return pool\\n\\n\\ndef corrupt_visible(rng, inst, roles):\\n    \\"\\"\\"One textual rule violation; returns (inst, kind) or None.\\"\\"\\"\\n    rec = copy.deepcopy(inst[\'rec\'])\\n    n = rec[\'n\']\\n    done = [qd for qd in rec[\'quests\'] if qd.get(\'result\') is not None]\\n    reasoning = [\'two_fail\', \'evil_overflow\', \'self_fail\', \'percival_three\']\\n    obvious = [\'fails_over_team\', \'team_size\']\\n    rng.shuffle(reasoning)\\n    rng.shuffle(obvious)\\n    kinds = reasoning + obvious if rng.random() < 0.7 else obvious + reasoning\\n    view = inst.get(\'view\')\\n    for kind in kinds:\\n        if kind == \'self_fail\' and view and view[\'role\'] not in EVIL_ROLES:\\n            mine = [qd for qd in done if view[\'player\'] in qd[\'team\']]\\n            if mine:\\n                qd = rng.choice(mine)\\n                qd[\'fails\'] = len(qd[\'team\'])\\n                qd[\'result\'] = \'fail\' if qd[\'fails\'] >= fails_required(n, qd[\'q\']) else \'success\'\\n                return {**inst, \'rec\': rec}, kind\\n        if kind == \'percival_three\' and not (view and view[\'role\'] == \'percival\'):\\n            continue\\n        if kind == \'fails_over_team\' and done:\\n            qd = rng.choice(done)\\n            qd[\'fails\'] = len(qd[\'team\']) + 1\\n            qd[\'result\'] = \'fail\'\\n            return {**inst, \'rec\': rec}, kind\\n        if kind == \'two_fail\' and n >= 7 and len(done) >= 4:\\n            qd = done[3]\\n            qd[\'fails\'], qd[\'result\'] = 1, \'fail\'\\n            return {**inst, \'rec\': rec}, kind\\n        if kind == \'evil_overflow\' and len(done) >= 2:\\n            a, b = done[0], done[1]\\n            if not set(a[\'team\']) & set(b[\'team\']):\\n                a[\'fails\'], b[\'fails\'] = len(a[\'team\']), len(b[\'team\'])\\n                if a[\'fails\'] + b[\'fails\'] > EVIL_COUNT[n]:\\n                    a[\'result\'] = \'fail\' if a[\'fails\'] >= fails_required(n, a[\'q\']) else \'success\'\\n                    b[\'result\'] = \'fail\' if b[\'fails\'] >= fails_required(n, b[\'q\']) else \'success\'\\n                    return {**inst, \'rec\': rec}, kind\\n        if kind == \'team_size\' and done:\\n            qd = rng.choice(done)\\n            extra = [i for i in range(n) if i not in qd[\'team\']]\\n            qd[\'team\'] = sorted(qd[\'team\'] + extra[:1])\\n            if qd[\'proposals\']:\\n                qd[\'proposals\'][-1][\'team\'] = list(qd[\'team\'])\\n            return {**inst, \'rec\': rec}, kind\\n        if kind == \'percival_three\':\\n            v = copy.deepcopy(inst[\'view\'])\\n            extra = [j for j in range(n) if j != v[\'player\'] and j not in v[\'seen\']]\\n            v[\'seen\'] = sorted(v[\'seen\'] + extra[:1])\\n            return {**inst, \'view\': v}, kind\\n    return None\\n\\n\\ndef to_text(inst):\\n    return render_record(inst[\'rec\'], show_votes=True) + \'\\\\n\' + render_view(inst.get(\'view\'), inst[\'rec\'][\'config\'])\\n\\n\\ndef build(parsed_path, out_dir, seed=137, backend=None, max_states=4000):\\n    rng = random.Random(seed)\\n    games = [json.loads(x) for x in open(parsed_path, encoding=\'utf-8\') if x.strip()]\\n    games = [g for g in games if not O.record_violations(g[\'rec\'], {}) and g[\'rec\'][\'quests\']]\\n    if backend is None or backend.name == \'oracle\':\\n        backend = backend or O.Oracle()\\n        games = [g for g in games if g[\'rec\'][\'n\'] <= 8]\\n    rng.shuffle(games)\\n    items, have = [], Counter()\\n    used_games = set()\\n    for g in games[:max_states]:\\n        if all(have[k] >= QUOTA[k] for k in QUOTA):\\n            break\\n        rec, roles = g[\'rec\'], tuple(g[\'roles\'])\\n        done = [k for k, qd in enumerate(rec[\'quests\'], 1) if qd.get(\'result\') is not None]\\n        if not done:\\n            continue\\n        k = rng.choice(done)\\n        r = prefix(rec, k)\\n        r[\'source\'] = rec.get(\'source\')\\n        persp = None if rng.random() < 0.3 else rng.randrange(rec[\'n\'])\\n        view = None if persp is None else true_view(roles, persp)\\n        inst = {\'rec\': r, \'view\': view}\\n        gid = (rec.get(\'source\') or {}).get(\'game\', str(len(items)))\\n        if have[\'inconsistent\'] < QUOTA[\'inconsistent\'] and rng.random() < 0.15:\\n            bad = corrupt_visible(rng, inst, roles)\\n            if bad:\\n                binst, kind = bad\\n                claim = rng.choice(claim_pool(rng, binst, roles))\\n                if backend.classify(binst, claim)[\'label\'] == \'inconsistent\':\\n                    items.append(make_item(binst, claim, \'inconsistent\', gid, k, roles, kind))\\n                    have[\'inconsistent\'] += 1\\n                    used_games.add(gid)\\n                    continue\\n        for claim in claim_pool(rng, inst, roles):\\n            lab = backend.classify(inst, claim)[\'label\']\\n            if lab in QUOTA and have[lab] < QUOTA[lab]:\\n                item = make_item(inst, claim, lab, gid, k, roles, None)\\n                if lab == \'undetermined\':\\n                    cg = min_reveal_cegar(backend, inst, claim)\\n                    item[\'gold_reveal\'] = {\'status\': cg[\'status\'], \'size\': cg.get(\'size\'), \'set\': cg.get(\'set\')}\\n                items.append(item)\\n                have[lab] += 1\\n                used_games.add(gid)\\n                break\\n    for i, it in enumerate(items):\\n        it[\'id\'] = f\'av{i:04d}\'\\n    out = Path(out_dir)\\n    out.mkdir(parents=True, exist_ok=True)\\n    with open(out / \'tasks.jsonl\', \'w\', encoding=\'utf-8\') as f:\\n        for it in items:\\n            f.write(json.dumps(it, ensure_ascii=False) + \'\\\\n\')\\n    blob = (out / \'tasks.jsonl\').read_bytes()\\n    manifest = {\'items\': len(items), \'labels\': dict(Counter(it[\'gold\'] for it in items)),\\n                \'games\': len(used_games), \'seed\': seed, \'backend\': backend.name,\\n                \'n_dist\': dict(Counter(it[\'inst\'][\'rec\'][\'n\'] for it in items)),\\n                \'perspective\': dict(Counter(it[\'perspective\'] for it in items)),\\n                \'claim_types\': dict(Counter(it[\'claim\'][\'t\'] for it in items)),\\n                \'corruptions\': dict(Counter(it[\'corruption\'] for it in items if it[\'corruption\'])),\\n                \'reveal_status\': dict(Counter((it.get(\'gold_reveal\') or {}).get(\'status\') for it in items\\n                                              if it[\'gold\'] == \'undetermined\')),\\n                \'tasks_sha256\': hashlib.sha256(blob).hexdigest(), \'source\': str(parsed_path)}\\n    (out / \'manifest.json\').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding=\'utf-8\')\\n    return manifest\\n\\n\\ndef make_item(inst, claim, gold, gid, k, roles, corruption):\\n    view = inst.get(\'view\')\\n    return {\'game\': gid, \'after_quest\': k, \'perspective\': \'observer\' if view is None else view[\'role\'],\\n            \'inst\': {\'rec\': inst[\'rec\'], \'view\': view}, \'claim\': claim, \'gold\': gold,\\n            \'corruption\': corruption, \'text\': to_text(inst), \'claim_text\': render_claim(claim),\\n            \'true_roles\': list(roles)}\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\'--parsed\', default=str(ROOT / \'results\' / \'audit\' / \'parsed_games.jsonl\'))\\n    ap.add_argument(\'--out\', default=str(ROOT / \'data\' / \'llm\'))\\n    ap.add_argument(\'--seed\', type=int, default=137)\\n    ap.add_argument(\'--backend\', default=\'auto\')\\n    a = ap.parse_args()\\n    backend = None\\n    if a.backend in (\'auto\', \'z3\'):\\n        try:\\n            from smt import SMT\\n            backend = SMT()\\n        except ImportError:\\n            if a.backend == \'z3\':\\n                raise\\n    print(json.dumps(build(a.parsed, a.out, a.seed, backend), ensure_ascii=False, indent=2))\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "avalon/llm_run.py": "\\"\\"\\"LLM experiment on frozen Avalon tasks (runs on a Colab A100).\\n\\nConditions (same items, same rule card):\\n  direct    : Qwen3-8B, thinking off, greedy, one-word verdict\\n  think     : Qwen3-8B, thinking on (sampling as recommended for thinking mode)\\n  extract   : the model only translates the text into the formal JSON; Z3 decides\\n  revise_*  : wrong direct answers get solver feedback and one more turn\\n              flag     = \\"your answer is not supported\\" only\\n              evidence = plus two consistent worlds or the minimal conflicting rules\\n  reveal    : for undetermined items, the model proposes Lady-of-the-Lake checks;\\n              scored for sufficiency (two-world test) and minimality\\n\\"\\"\\"\\nimport hashlib\\nimport json\\nimport random\\nimport re\\nimport time\\nfrom collections import Counter, defaultdict\\nfrom pathlib import Path\\n\\nfrom game import ZH_ROLE, ROLES, EVIL_ROLES, TEAM_SIZE, fails_required, player_label, team_label\\n\\nMODEL = \'Qwen/Qwen3-8B\'\\nREVISION = \'b968826d9c46dd6066d109eabc6255188de91218\'\\nLABELS = (\'entailed\', \'contradicted\', \'undetermined\', \'inconsistent\')\\nWORD = {\'entailed\': \'成立\', \'contradicted\': \'不成立\', \'undetermined\': \'不确定\', \'inconsistent\': \'矛盾\'}\\nFROM_WORD = {v: k for k, v in WORD.items()}\\n\\nRULE_CARD = \'\'\'你在分析一局《阿瓦隆》（The Resistance: Avalon）。只能依据下列规则和给出的记录做严格的逻辑推理；发言、投票倾向、\\"像不像\\"之类的策略性推测都不算证据。\\n\\n【阵营与人数】好人阵营：梅林、派西维尔、亚瑟的忠臣。坏人阵营：莫甘娜、莫德雷德、奥伯伦、莫德雷德的爪牙。坏人人数：5人局2名，6人局2名，7人局3名，8人局3名，9人局3名，10人局4名。本局使用的角色在记录开头给出，每人一张。\\n【开局信息】每个人知道自己的角色。除奥伯伦外的坏人互相认识，他们看不到奥伯伦，奥伯伦也看不到他们。梅林能看到除莫德雷德以外的所有坏人（包括奥伯伦）。派西维尔能看到梅林和莫甘娜，但分不清谁是谁。亚瑟的忠臣和奥伯伦开局得不到任何信息。\\n【组队与投票】队长提名队伍，所有人公开投票，赞成票严格过半才通过（平票算否决）。每次提案后队长按座位顺序交给下一位。同一轮连续5次否决则坏人直接获胜。投票可以随意投，规则上与身份无关。各轮出队人数：5人局2,3,2,3,3；6人局2,3,4,3,4；7人局2,3,3,4,4；8至10人局3,4,4,5,5。\\n【执行任务】每名队员出一张任务牌：好人必须出成功；坏人可以出成功，也可以出失败。牌洗混后翻开，所有人知道失败票的张数，但不知道是谁出的。出现1张失败票任务就失败；例外：7人及以上的第4个任务需要至少2张失败票才失败。\\n【结束】三个任务失败则坏人获胜。三个任务成功后刺客刺杀一人：刺中梅林则坏人获胜，否则好人获胜。\\n【湖中仙女】持有者查验一名玩家，得知其阵营（好人或坏人），结果只有持有者本人知道。\'\'\'\\n\\nTASK = \'\'\'判断给定\\"说法\\"相对于规则和记录（以及你的视角信息）的地位，只能回答一个词：\\n成立：在所有符合规则与记录的身份分配中，说法都为真；\\n不成立：在所有这样的身份分配中，说法都为假；\\n不确定：有的身份分配下为真，有的为假；\\n矛盾：记录或你的视角信息本身违反规则，不存在任何符合的身份分配。\'\'\'\\n\\nANSWER = \'只输出 成立、不成立、不确定、矛盾 中的一个词，不要解释。\'\\n\\nEXTRACT_SYS = \'\'\'把一段《阿瓦隆》局面描述逐字转换成JSON，不做任何推理。格式：\\n{\\"n\\": 人数,\\n \\"config\\": {角色英文名: 数量, ...}（只列出现的角色）,\\n \\"quests\\": [{\\"q\\": 任务编号, \\"team\\": [座位号], \\"fails\\": 失败票张数, \\"result\\": \\"success\\"或\\"fail\\"}]（只列已执行的任务；提案和投票不需要）,\\n \\"view\\": null 或 {\\"player\\": 座位号, \\"role\\": 角色英文名, \\"seen\\": [开局看到的座位号], \\"lady\\": [[被查验的座位号, 是坏人则true否则false]]},\\n \\"claim\\": 说法的JSON}\\n说法的写法：X号一定是坏人 → {\\"t\\":\\"evil\\",\\"p\\":X}；X号一定是好人 → {\\"t\\":\\"good\\",\\"p\\":X}；X号一定是梅林 → {\\"t\\":\\"role\\",\\"p\\":X,\\"r\\":\\"merlin\\"}；X号一定不是梅林 → {\\"t\\":\\"not\\",\\"a\\":{\\"t\\":\\"role\\",\\"p\\":X,\\"r\\":\\"merlin\\"}}；某些人中至少有k名坏人 → {\\"t\\":\\"atleast\\",\\"ps\\":[座位号],\\"k\\":k}；某些人中恰好有k名坏人 → {\\"t\\":\\"exactly\\",\\"ps\\":[座位号],\\"k\\":k}；X号和Y号一定属于同一阵营 → {\\"t\\":\\"same\\",\\"ps\\":[X,Y]}；本局的坏人恰好是某些人 → {\\"t\\":\\"evil_set\\",\\"ps\\":[座位号]}。\\n角色英文名：梅林 merlin，派西维尔 percival，亚瑟的忠臣 servant，莫甘娜 morgana，莫德雷德 mordred，奥伯伦 oberon，莫德雷德的爪牙 minion。座位号使用文字中的号码（从1开始）。旁观者视角时 view 为 null。只输出一个JSON对象，不要解释，不要Markdown。\'\'\'\\n\\nEXTRACT_EXAMPLE_IN = \'\'\'6人局，角色为：梅林、派西维尔、亚瑟的忠臣×2、莫甘娜、莫德雷德的爪牙（共2名坏人）。\\n第1轮第1次提案：队长3号提名1号、6号，赞成者为1号、3号、6号，被否决。\\n第1轮第2次提案：队长4号提名2号、4号，赞成者为1号、2号、4号、5号，通过。\\n第1个任务由2号、4号执行，翻开1张失败票，任务失败。\\n你是5号，身份是派西维尔。开局时你看到1号、3号（其中一人是梅林，另一人是莫甘娜，你分不清）。\\n说法：2号和4号中至少有1名坏人。\'\'\'\\nEXTRACT_EXAMPLE_OUT = (\'{\\"n\\": 6, \\"config\\": {\\"merlin\\": 1, \\"percival\\": 1, \\"servant\\": 2, \\"morgana\\": 1, \\"minion\\": 1}, \'\\n                       \'\\"quests\\": [{\\"q\\": 1, \\"team\\": [2, 4], \\"fails\\": 1, \\"result\\": \\"fail\\"}], \'\\n                       \'\\"view\\": {\\"player\\": 5, \\"role\\": \\"percival\\", \\"seen\\": [1, 3], \\"lady\\": []}, \'\\n                       \'\\"claim\\": {\\"t\\": \\"atleast\\", \\"ps\\": [2, 4], \\"k\\": 1}}\')\\n\\nREVEAL_Q = (\'现在可以用湖中仙女查验若干玩家的阵营（每查验一人，就得知他是好人还是坏人）。\'\\n            \'请一次性给出最少需要查验的玩家，使得无论查验结果如何，都一定能判断上述说法成立还是不成立。\'\\n            \'不能查验你自己。只输出座位号的JSON数组，例如 [3, 5]，不要解释。\')\\n\\n\\ndef digest(x):\\n    return hashlib.sha256(json.dumps(x, ensure_ascii=False, sort_keys=True).encode()).hexdigest()\\n\\n\\ndef user_text(item):\\n    return item[\'text\'] + \'\\\\n\\\\n说法：\' + item[\'claim_text\']\\n\\n\\ndef job(item, task, messages, budget, thinking=False):\\n    return {\'id\': f\'{task}/{item[\\"id\\"]}\', \'item\': item[\'id\'], \'task\': task, \'messages\': messages,\\n            \'max_new_tokens\': budget, \'thinking\': thinking}\\n\\n\\ndef initial_jobs(items):\\n    jobs = []\\n    sys_msg = RULE_CARD + \'\\\\n\\\\n\' + TASK\\n    for it in items:\\n        base = [{\'role\': \'system\', \'content\': sys_msg}, {\'role\': \'user\', \'content\': user_text(it) + \'\\\\n\' + ANSWER}]\\n        jobs.append(job(it, \'direct\', base, 16))\\n        jobs.append(job(it, \'extract\', [{\'role\': \'system\', \'content\': EXTRACT_SYS},\\n                                        {\'role\': \'user\', \'content\': EXTRACT_EXAMPLE_IN},\\n                                        {\'role\': \'assistant\', \'content\': EXTRACT_EXAMPLE_OUT},\\n                                        {\'role\': \'user\', \'content\': user_text(it)}], 768))\\n        if it[\'gold\'] == \'undetermined\' and (it.get(\'gold_reveal\') or {}).get(\'status\') == \'ok\':\\n            jobs.append(job(it, \'reveal\', [{\'role\': \'system\', \'content\': RULE_CARD},\\n                                           {\'role\': \'user\', \'content\': user_text(it) + \'\\\\n\' + REVEAL_Q}], 64))\\n    return jobs\\n\\n\\ndef think_jobs(items, budget=3072):\\n    sys_msg = RULE_CARD + \'\\\\n\\\\n\' + TASK\\n    return [job(it, \'think\', [{\'role\': \'system\', \'content\': sys_msg},\\n                              {\'role\': \'user\', \'content\': user_text(it) + \'\\\\n\' + ANSWER}], budget, thinking=True)\\n            for it in items]\\n\\n\\n# ---------------------------------------------------------------------------\\n# parsing\\n# ---------------------------------------------------------------------------\\ndef strip_think(raw):\\n    return raw.split(\'</think>\')[-1] if \'</think>\' in raw else raw\\n\\n\\ndef parse_label(raw):\\n    t = strip_think(raw).strip().strip(\'。.`*\\"“”\\\\\' \\\\n\')\\n    if t in FROM_WORD:\\n        return FROM_WORD[t], \'strict\'\\n    found = set()\\n    for w in (\'不成立\', \'不确定\', \'矛盾\'):\\n        if w in t:\\n            found.add(FROM_WORD[w])\\n    if \'成立\' in t.replace(\'不成立\', \'\'):\\n        found.add(\'entailed\')\\n    if len(found) == 1:\\n        return found.pop(), \'lenient\'\\n    return \'parse_failure\', None\\n\\n\\ndef _json_block(raw):\\n    t = strip_think(raw).strip()\\n    t = re.sub(r\'^```(?:json)?|```$\', \'\', t.strip(), flags=re.M).strip()\\n    start = t.find(\'{\') if \'{\' in t else t.find(\'[\')\\n    if start < 0:\\n        raise ValueError(\'no json\')\\n    dec = json.JSONDecoder()\\n    obj, _ = dec.raw_decode(t[start:])\\n    return obj\\n\\n\\ndef _claim_to0(c):\\n    c = dict(c)\\n    if c.get(\'t\') in (\'evil\', \'good\', \'role\'):\\n        c[\'p\'] = int(c[\'p\']) - 1\\n    elif c.get(\'t\') in (\'atleast\', \'exactly\', \'same\', \'evil_set\'):\\n        c[\'ps\'] = sorted(int(x) - 1 for x in c[\'ps\'])\\n        if \'k\' in c:\\n            c[\'k\'] = int(c[\'k\'])\\n    elif c.get(\'t\') == \'not\':\\n        c[\'a\'] = _claim_to0(c[\'a\'])\\n    else:\\n        raise ValueError(\'claim type\')\\n    return c\\n\\n\\ndef parse_extraction(raw):\\n    \\"\\"\\"Model JSON (1-based seats) -> (inst, claim) with 0-based seats.\\"\\"\\"\\n    x = _json_block(raw)\\n    n = int(x[\'n\'])\\n    cfg = {str(k): int(v) for k, v in x[\'config\'].items() if int(v) > 0}\\n    if any(k not in ROLES for k in cfg):\\n        raise ValueError(\'role name\')\\n    quests = []\\n    for q in x.get(\'quests\') or []:\\n        res = str(q[\'result\']).lower()\\n        if res not in (\'success\', \'fail\'):\\n            raise ValueError(\'result\')\\n        quests.append({\'q\': int(q[\'q\']), \'team\': sorted(int(i) - 1 for i in q[\'team\']),\\n                       \'fails\': None if q.get(\'fails\') is None else int(q[\'fails\']), \'result\': res, \'proposals\': []})\\n    v = x.get(\'view\')\\n    view = None\\n    if v:\\n        view = {\'player\': int(v[\'player\']) - 1, \'role\': str(v[\'role\']),\\n                \'seen\': sorted(int(i) - 1 for i in v.get(\'seen\') or []),\\n                \'lady\': [[int(t) - 1, bool(e)] for t, e in v.get(\'lady\') or []]}\\n        if view[\'role\'] not in ROLES:\\n            raise ValueError(\'view role\')\\n    inst = {\'rec\': {\'n\': n, \'config\': cfg, \'quests\': quests, \'end\': None}, \'view\': view}\\n    return inst, _claim_to0(x[\'claim\'])\\n\\n\\ndef parse_reveal(raw, cands):\\n    x = _json_block(raw)\\n    if not isinstance(x, list) or any(type(i) is not int for i in x):\\n        raise ValueError(\'not an int array\')\\n    s = sorted(i - 1 for i in x)\\n    if len(set(s)) != len(s):\\n        raise ValueError(\'duplicate\')\\n    if any(i not in cands for i in s):\\n        raise ValueError(\'not a candidate\')\\n    return s\\n\\n\\ndef gold_shape(item):\\n    \\"\\"\\"Extraction target: the item\'s formalisation without proposals.\\"\\"\\"\\n    rec = item[\'inst\'][\'rec\']\\n    quests = [{\'q\': q[\'q\'], \'team\': sorted(q[\'team\']), \'fails\': q[\'fails\'], \'result\': q[\'result\']}\\n              for q in rec[\'quests\'] if q.get(\'result\') is not None]\\n    return {\'n\': rec[\'n\'], \'config\': {k: v for k, v in rec[\'config\'].items() if v}, \'quests\': quests,\\n            \'view\': item[\'inst\'][\'view\'], \'claim\': item[\'claim\']}\\n\\n\\ndef shape_of(inst, claim):\\n    rec = inst[\'rec\']\\n    return {\'n\': rec[\'n\'], \'config\': rec[\'config\'],\\n            \'quests\': [{\'q\': q[\'q\'], \'team\': q[\'team\'], \'fails\': q[\'fails\'], \'result\': q[\'result\']} for q in rec[\'quests\']],\\n            \'view\': inst[\'view\'], \'claim\': claim}\\n\\n\\ndef _norm_view(v):\\n    if v is None:\\n        return None\\n    return {\'player\': v[\'player\'], \'role\': v[\'role\'], \'seen\': sorted(v.get(\'seen\') or []),\\n            \'lady\': sorted([list(map(int, x[:1])) + [bool(x[1])] for x in v.get(\'lady\') or []])}\\n\\n\\ndef _norm_claim(c):\\n    c = dict(c)\\n    if \'ps\' in c:\\n        c[\'ps\'] = sorted(c[\'ps\'])\\n    if c.get(\'t\') == \'not\':\\n        c[\'a\'] = _norm_claim(c[\'a\'])\\n    return c\\n\\n\\n# ---------------------------------------------------------------------------\\n# solver feedback\\n# ---------------------------------------------------------------------------\\ndef render_world(roles):\\n    return \'，\'.join(f\'{player_label(i)}{ZH_ROLE[r]}\' for i, r in enumerate(roles))\\n\\n\\ndef render_core(tags, item):\\n    rec = item[\'inst\'][\'rec\']\\n    cfg, n = rec[\'config\'], rec[\'n\']\\n    out = []\\n    for tag in tags:\\n        rid, _, where = tag.partition(\'@\')\\n        if rid.startswith(\'S-DEAL\'):\\n            r = rid[rid.index(\'[\') + 1:-1]\\n            out.append(f\'本局{ZH_ROLE[r]}共{cfg.get(r, 0)}名\')\\n        elif rid == \'Q-COUNT\':\\n            q = int(where[1:])\\n            f = next(x[\'fails\'] for x in rec[\'quests\'] if x[\'q\'] == q)\\n            out.append(f\'第{q}个任务翻开了{f}张失败票\')\\n        elif rid == \'Q-GOOD\':\\n            q, i = where[1:].split(\':\')\\n            out.append(f\'好人必须出成功票（第{q}个任务中的{int(i) + 1}号）\')\\n        elif rid == \'Q-RESULT\':\\n            q = int(where[1:])\\n            out.append(f\'第{q}个任务需要至少{fails_required(n, q)}张失败票才算失败\')\\n        elif rid == \'Q-SIZE\':\\n            q = int(where[1:])\\n            out.append(f\'第{q}个任务应由{TEAM_SIZE[n][q - 1]}人执行\')\\n        elif rid == \'V-SELF\':\\n            out.append(\'你自己的身份\')\\n        elif rid in (\'K-MERLIN\', \'K-PERCIVAL\', \'K-EVIL\', \'K-NONE\'):\\n            who = {\'K-MERLIN\': \'梅林\', \'K-PERCIVAL\': \'派西维尔\', \'K-EVIL\': \'坏人\', \'K-NONE\': \'该身份\'}[rid]\\n            out.append(f\'{who}的开局视野（关于{int(where) + 1}号）\')\\n        elif rid == \'K-SELF\':\\n            out.append(\'开局视野不包括自己\')\\n        elif rid == \'K-LADY\':\\n            out.append(f\'湖中仙女对{int(where) + 1}号的查验结果\')\\n        elif rid == \'S-COUNT\':\\n            out.append(\'角色配置中的坏人数量必须符合人数表\')\\n        else:\\n            out.append(f\'规则{rid}\' + (f\'（{where}）\' if where else \'\'))\\n    return \'；\'.join(out) + \'。\'\\n\\n\\ndef feedback(item, answer_label, smt, mode):\\n    head = f\'约束求解器检查了你的回答“{WORD.get(answer_label, answer_label)}”，发现它不被规则与记录支持。\'\\n    if mode == \'flag\':\\n        return head + \'请重新判断。\' + ANSWER\\n    res = smt.classify(item[\'inst\'], item[\'claim\'], explain=True)\\n    g = res[\'label\']\\n    if g == \'undetermined\':\\n        ev = (\'下面两种身份分配都符合规则与记录：\\\\n情形一（说法为真）：\' + render_world(res[\'positive\'])\\n              + \'\\\\n情形二（说法为假）：\' + render_world(res[\'negative\']))\\n    elif g == \'inconsistent\':\\n        ev = \'记录本身违反规则，冲突来自：\' + render_core(res[\'core\'], item)\\n    else:\\n        w = res[\'positive\'] or res[\'negative\']\\n        ev = (\'存在符合规则与记录的身份分配，例如：\' + render_world(w) + \'\\\\n\') if answer_label == \'inconsistent\' else \'\'\\n        ev += (\'不存在使说法为假的身份分配\' if g == \'entailed\' else \'不存在使说法为真的身份分配\')\\n        ev += \'，推理所需的规则与记录：\' + render_core(res[\'core\'], item)\\n    return head + \'\\\\n\' + ev + \'\\\\n请重新判断。\' + ANSWER\\n\\n\\ndef revise_jobs(items, direct_records, smt, mode):\\n    by = {r[\'item\']: r for r in direct_records}\\n    jobs = []\\n    sys_msg = RULE_CARD + \'\\\\n\\\\n\' + TASK\\n    for it in items:\\n        r = by[it[\'id\']]\\n        lab, _ = parse_label(r[\'raw\'])\\n        if lab == it[\'gold\']:\\n            continue\\n        msgs = [{\'role\': \'system\', \'content\': sys_msg},\\n                {\'role\': \'user\', \'content\': user_text(it) + \'\\\\n\' + ANSWER},\\n                {\'role\': \'assistant\', \'content\': r[\'raw\'].strip() or \'（无输出）\'},\\n                {\'role\': \'user\', \'content\': feedback(it, lab, smt, mode)}]\\n        jobs.append(job(it, \'revise_\' + mode, msgs, 16))\\n    return jobs\\n\\n\\n# ---------------------------------------------------------------------------\\n# model runner\\n# ---------------------------------------------------------------------------\\nclass Runner:\\n    def __init__(self, out_dir, batch_size=None):\\n        import torch\\n        import transformers\\n        import platform\\n        from transformers import AutoTokenizer, AutoModelForCausalLM\\n        assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), \'Need a BF16-capable CUDA GPU runtime\'\\n        self.torch = torch\\n        mem = torch.cuda.get_device_properties(0).total_memory / 2 ** 30\\n        self.batch = batch_size or (16 if mem > 60 else 8)\\n        self.tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, padding_side=\'left\')\\n        if self.tok.pad_token_id is None:\\n            self.tok.pad_token = self.tok.eos_token\\n        self.model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, torch_dtype=torch.bfloat16,\\n                                                          attn_implementation=\'sdpa\').to(\'cuda\').eval()\\n        self.out = Path(out_dir)\\n        self.out.mkdir(parents=True, exist_ok=True)\\n        self.meta = {\'model\': MODEL, \'revision\': REVISION, \'gpu\': torch.cuda.get_device_name(0),\\n                     \'gpu_mem_gib\': round(mem, 1), \'python\': platform.python_version(), \'torch\': torch.__version__,\\n                     \'transformers\': transformers.__version__, \'dtype\': \'bfloat16\', \'batch_size\': self.batch,\\n                     \'greedy_when_thinking_off\': True,\\n                     \'thinking_sampling\': {\'temperature\': 0.6, \'top_p\': 0.95, \'top_k\': 20},\\n                     \'seed\': 42, \'code_sha256\': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}\\n        (self.out / \'runner_metadata.json\').write_text(json.dumps(self.meta, indent=2), encoding=\'utf-8\')\\n\\n    def run(self, jobs, name):\\n        meta = {\'jobs_sha256\': digest(jobs), \'n\': len(jobs)}\\n        mp, rp = self.out / f\'{name}_metadata.json\', self.out / f\'{name}_records.jsonl\'\\n        if mp.exists():\\n            assert json.loads(mp.read_text()) == meta, \'resume mismatch: use a new name\'\\n        else:\\n            mp.write_text(json.dumps(meta, indent=2), encoding=\'utf-8\')\\n        (self.out / f\'{name}_jobs.json\').write_text(json.dumps(jobs, ensure_ascii=False), encoding=\'utf-8\')\\n        records = [json.loads(x) for x in rp.read_text(encoding=\'utf-8\').splitlines() if x.strip()] if rp.exists() else []\\n        done = {r[\'id\'] for r in records}\\n        groups = sorted({(j[\'max_new_tokens\'], j[\'thinking\']) for j in jobs})\\n        for budget, thinking in groups:\\n            todo = [j for j in jobs if j[\'id\'] not in done and j[\'max_new_tokens\'] == budget and j[\'thinking\'] == thinking]\\n            bs = self.batch if budget <= 1024 else max(4, self.batch)\\n            for b0 in range(0, len(todo), bs):\\n                batch = todo[b0:b0 + bs]\\n                texts = [self.tok.apply_chat_template(j[\'messages\'], tokenize=False, add_generation_prompt=True,\\n                                                      enable_thinking=thinking) for j in batch]\\n                enc = self.tok(texts, return_tensors=\'pt\', padding=True).to(\'cuda\')\\n                kw = dict(max_new_tokens=budget, pad_token_id=self.tok.pad_token_id)\\n                if thinking:\\n                    self.torch.manual_seed(42 + b0)\\n                    kw.update(do_sample=True, temperature=0.6, top_p=0.95, top_k=20)\\n                else:\\n                    kw.update(do_sample=False)\\n                self.torch.cuda.synchronize()\\n                t0 = time.perf_counter()\\n                with self.torch.inference_mode():\\n                    out = self.model.generate(**enc, **kw)\\n                self.torch.cuda.synchronize()\\n                sec = time.perf_counter() - t0\\n                gen = out[:, enc.input_ids.shape[1]:]\\n                dec = self.tok.batch_decode(gen, skip_special_tokens=True)\\n                with rp.open(\'a\', encoding=\'utf-8\') as f:\\n                    for j, raw, ids in zip(batch, dec, gen):\\n                        ids = ids.tolist()\\n                        eos = self.tok.eos_token_id\\n                        row = {\'id\': j[\'id\'], \'item\': j[\'item\'], \'task\': j[\'task\'], \'raw\': raw,\\n                               \'output_tokens\': ids.index(eos) + 1 if eos in ids else len(ids),\\n                               \'hit_limit\': eos not in ids, \'batch_seconds\': sec, \'batch_size\': len(batch),\\n                               \'input_tokens\': int(enc.input_ids.shape[1])}\\n                        f.write(json.dumps(row, ensure_ascii=False) + \'\\\\n\')\\n                        records.append(row)\\n                print(f\'{name} budget={budget} think={thinking}: {b0 + len(batch)}/{len(todo)} {sec:.1f}s\', flush=True)\\n        assert len({r[\'id\'] for r in records}) == len(jobs)\\n        return records\\n\\n\\n# ---------------------------------------------------------------------------\\n# scoring\\n# ---------------------------------------------------------------------------\\ndef cls_metrics(rows):\\n    n = len(rows)\\n    conf = {g: Counter() for g in LABELS}\\n    for r in rows:\\n        conf[r[\'gold\']][r[\'pred\']] += 1\\n    f1 = []\\n    for k in LABELS:\\n        tp = conf[k][k]\\n        fp = sum(conf[g][k] for g in LABELS if g != k)\\n        fn = sum(v for p, v in conf[k].items() if p != k)\\n        f1.append(2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0)\\n    und = [r for r in rows if r[\'gold\'] == \'undetermined\']\\n    dec = [r for r in rows if r[\'gold\'] in (\'entailed\', \'contradicted\')]\\n    inc = [r for r in rows if r[\'gold\'] == \'inconsistent\']\\n    return {\'n\': n, \'accuracy\': sum(r[\'gold\'] == r[\'pred\'] for r in rows) / n if n else None,\\n            \'macro_f1\': sum(f1) / 4, \'f1\': dict(zip(LABELS, f1)),\\n            \'confusion\': {g: dict(c) for g, c in conf.items()},\\n            \'premature_certainty\': [sum(r[\'pred\'] in (\'entailed\', \'contradicted\') for r in und), len(und)],\\n            \'false_uncertainty\': [sum(r[\'pred\'] == \'undetermined\' for r in dec), len(dec)],\\n            \'missed_inconsistency\': [sum(r[\'pred\'] != \'inconsistent\' for r in inc), len(inc)],\\n            \'false_inconsistency\': [sum(r[\'pred\'] == \'inconsistent\' for r in rows if r[\'gold\'] != \'inconsistent\'),\\n                                    sum(r[\'gold\'] != \'inconsistent\' for r in rows)],\\n            \'parse_failures\': sum(r[\'pred\'] == \'parse_failure\' for r in rows)}\\n\\n\\ndef bootstrap_diff(a, b, seed=271, reps=2000):\\n    \\"\\"\\"Paired bootstrap over items of mean(b) - mean(a); a, b are 0/1 lists.\\"\\"\\"\\n    rng = random.Random(seed)\\n    n = len(a)\\n    diffs = []\\n    for _ in range(reps):\\n        idx = [rng.randrange(n) for _ in range(n)]\\n        diffs.append(sum(b[i] - a[i] for i in idx) / n)\\n    diffs.sort()\\n    return {\'diff\': sum(b[i] - a[i] for i in range(n)) / n, \'ci95\': [diffs[int(0.025 * reps)], diffs[int(0.975 * reps) - 1]]}\\n\\n\\ndef strata(items):\\n    def pkind(p):\\n        return \'observer\' if p == \'observer\' else (\'evil_view\' if p in EVIL_ROLES else \'good_view\')\\n    return {it[\'id\']: {\'perspective\': pkind(it[\'perspective\']), \'claim\': it[\'claim\'][\'t\'],\\n                       \'n\': it[\'inst\'][\'rec\'][\'n\'], \'corruption\': it.get(\'corruption\') or \'-\'} for it in items}\\n\\n\\ndef score(items, recs, smt):\\n    by = defaultdict(dict)\\n    for task, rr in recs.items():\\n        for r in rr:\\n            by[task][r[\'item\']] = r\\n    st = strata(items)\\n    out, correct = {}, {}\\n    for task in (\'direct\', \'think\'):\\n        if task not in by:\\n            continue\\n        rows = []\\n        for it in items:\\n            lab, mode = parse_label(by[task][it[\'id\']][\'raw\'])\\n            rows.append({\'id\': it[\'id\'], \'gold\': it[\'gold\'], \'pred\': lab, \'mode\': mode})\\n        m = cls_metrics(rows)\\n        m[\'lenient_parses\'] = sum(r[\'mode\'] == \'lenient\' for r in rows)\\n        m[\'hit_token_limit\'] = sum(by[task][it[\'id\']].get(\'hit_limit\', False) for it in items)\\n        m[\'mean_output_tokens\'] = sum(by[task][it[\'id\']][\'output_tokens\'] for it in items) / len(items)\\n        m[\'by\'] = {k: {v: cls_metrics([r for r in rows if st[r[\'id\']][k] == v])[\'accuracy\']\\n                       for v in sorted({s[k] for s in st.values()}, key=str)} for k in (\'perspective\', \'claim\', \'n\')}\\n        out[task] = m\\n        correct[task] = [int(r[\'gold\'] == r[\'pred\']) for r in rows]\\n    # extraction + solver\\n    if \'extract\' in by:\\n        rows, ex = [], Counter()\\n        for it in items:\\n            raw = by[\'extract\'][it[\'id\']][\'raw\']\\n            try:\\n                inst, claim = parse_extraction(raw)\\n                ex[\'valid_json\'] += 1\\n                gs, ps = gold_shape(it), shape_of(inst, claim)\\n                ex[\'n_ok\'] += gs[\'n\'] == ps[\'n\']\\n                ex[\'config_ok\'] += gs[\'config\'] == ps[\'config\']\\n                ex[\'quests_ok\'] += gs[\'quests\'] == ps[\'quests\']\\n                ex[\'view_ok\'] += _norm_view(gs[\'view\']) == _norm_view(ps[\'view\'])\\n                ex[\'claim_ok\'] += _norm_claim(gs[\'claim\']) == _norm_claim(ps[\'claim\'])\\n                exact = (gs[\'n\'] == ps[\'n\'] and gs[\'config\'] == ps[\'config\'] and gs[\'quests\'] == ps[\'quests\']\\n                         and _norm_view(gs[\'view\']) == _norm_view(ps[\'view\']) and _norm_claim(gs[\'claim\']) == _norm_claim(ps[\'claim\']))\\n                ex[\'exact\'] += exact\\n                try:\\n                    lab = smt.classify(inst, claim)[\'label\']\\n                except Exception:\\n                    lab = \'parse_failure\'\\n                    ex[\'solver_error\'] += 1\\n            except Exception:\\n                lab = \'parse_failure\'\\n            rows.append({\'id\': it[\'id\'], \'gold\': it[\'gold\'], \'pred\': lab})\\n        m = cls_metrics(rows)\\n        m[\'extraction\'] = dict(ex)\\n        m[\'by\'] = {k: {v: cls_metrics([r for r in rows if st[r[\'id\']][k] == v])[\'accuracy\']\\n                       for v in sorted({s[k] for s in st.values()}, key=str)} for k in (\'perspective\', \'claim\', \'n\')}\\n        out[\'extract_solve\'] = m\\n        correct[\'extract_solve\'] = [int(r[\'gold\'] == r[\'pred\']) for r in rows]\\n    # revision with solver feedback (only wrong direct answers were re-asked)\\n    for mode in (\'flag\', \'evidence\'):\\n        t = \'revise_\' + mode\\n        if t not in by:\\n            continue\\n        rows, fixed = [], 0\\n        for it in items:\\n            lab, _ = parse_label(by[\'direct\'][it[\'id\']][\'raw\'])\\n            if lab != it[\'gold\']:\\n                lab2, _ = parse_label(by[t][it[\'id\']][\'raw\'])\\n                fixed += lab2 == it[\'gold\']\\n                lab = lab2\\n            rows.append({\'id\': it[\'id\'], \'gold\': it[\'gold\'], \'pred\': lab})\\n        m = cls_metrics(rows)\\n        m[\'reasked\'] = len(by[t])\\n        m[\'fixed\'] = fixed\\n        out[t] = m\\n        correct[t] = [int(r[\'gold\'] == r[\'pred\']) for r in rows]\\n    # reveal selection\\n    if \'reveal\' in by:\\n        from search import candidates\\n        rv = Counter()\\n        excess = []\\n        for it in items:\\n            if it[\'id\'] not in by[\'reveal\']:\\n                continue\\n            rv[\'n\'] += 1\\n            cands = candidates(it[\'inst\'])\\n            try:\\n                S = parse_reveal(by[\'reveal\'][it[\'id\']][\'raw\'], cands)\\n            except Exception:\\n                rv[\'invalid\'] += 1\\n                continue\\n            suff = smt.find_pair(it[\'inst\'], it[\'claim\'], S) is None\\n            rv[\'sufficient\'] += suff\\n            gsize = it[\'gold_reveal\'][\'size\']\\n            if suff:\\n                rv[\'optimal\'] += len(S) == gsize\\n                excess.append(len(S) - gsize)\\n            rv[\'all_candidates_chosen\'] += len(S) == len(cands)\\n        out[\'reveal\'] = {**rv, \'mean_excess_when_sufficient\': sum(excess) / len(excess) if excess else None,\\n                         \'gold_total\': sum(it[\'gold_reveal\'][\'size\'] for it in items if it[\'id\'] in by[\'reveal\'])}\\n    # end-to-end verification of direct answers with the model\'s own formalisation\\n    if \'direct\' in by and \'extract\' in by:\\n        flagged_wrong = flagged_right = wrong = right = 0\\n        for it in items:\\n            lab, _ = parse_label(by[\'direct\'][it[\'id\']][\'raw\'])\\n            try:\\n                inst, claim = parse_extraction(by[\'extract\'][it[\'id\']][\'raw\'])\\n                verdict = smt.classify(inst, claim)[\'label\']\\n            except Exception:\\n                verdict = None\\n            flag = verdict is not None and verdict != lab\\n            if lab == it[\'gold\']:\\n                right += 1\\n                flagged_right += flag\\n            else:\\n                wrong += 1\\n                flagged_wrong += flag\\n        out[\'verification_pipeline\'] = {\'direct_wrong\': wrong, \'wrong_flagged\': flagged_wrong,\\n                                         \'direct_right\': right, \'right_flagged\': flagged_right}\\n    comps = {}\\n    for t in (\'think\', \'extract_solve\', \'revise_flag\', \'revise_evidence\'):\\n        if t in correct and \'direct\' in correct:\\n            comps[f\'{t}_minus_direct\'] = bootstrap_diff(correct[\'direct\'], correct[t])\\n    out[\'paired_bootstrap\'] = comps\\n    return out\\n", "RULES.md": "# 阿瓦隆规则条款覆盖表\\n\\n材料：*The Resistance: Avalon* 官方规则书（Indie Boards & Cards，Don Eskridge 设计）。表中规则为本项目的转述，不是原文摘录。\\n核对来源：rulespal.com 的规则书整理页、avalon-game.com 规则 wiki（出队人数表）。两处来源的差异记在“解释选择”一节。\\n\\n实现：`avalon/smt.py`（Z3，按条款编号跟踪每个约束实例）与 `avalon/oracle.py`（独立穷举，表格重新录入，任务规则用另一种写法）。测试编号见 `avalon/cases.py`。\\n\\n## 1. 已形式化的条款\\n\\n| 编号 | 规则书章节 | 规则（转述） | 形式化 | 测试 |\\n|---|---|---|---|---|\\n| S-SIZE | Set Up | 5–10 人 | n ∈ {5,…,10} | 随机实例 |\\n| S-COUNT | Set Up | 好人/坏人人数表：5→3/2，6→4/2，7→4/3，8→5/3，9→6/3，10→6/4 | 配置中坏人角色数 = E(n) | T01–T03 |\\n| S-TOTAL | Set Up | 每人发一张角色牌 | 配置总数 = n | 篡改实例 |\\n| S-UNIQUE | Optional Characters | 梅林、派西维尔、莫甘娜、莫德雷德、奥伯伦各至多一张 | 特殊角色计数 ≤ 1 | — |\\n| S-DEAL[r] | Set Up | 角色按配置随机发放 | Σᵢ[roleᵢ=r] = count(r)，对每个角色 r | T01 |\\n| V-SELF | Set Up | 每人知道自己的角色 | role_p = 自身角色（视角事实） | T33, T35 |\\n| K-EVIL | Evil reveals itself | 坏人（奥伯伦除外）开局互相认识 | 对 j≠p：(evil_j ∧ role_j≠奥伯伦) ⇔ j∈seen | T31, T32 |\\n| K-MERLIN | Merlin looks into the future | 梅林看到坏人，莫德雷德除外 | 对 j≠p：(evil_j ∧ role_j≠莫德雷德 [∧ role_j≠奥伯伦]) ⇔ j∈seen | T26–T28, T43–T44 |\\n| K-PERCIVAL | Percival | 派西维尔看到梅林和莫甘娜，但分不清 | 对 j≠p：role_j∈{梅林,莫甘娜} ⇔ j∈seen | T29, T30, T34 |\\n| K-NONE | Set Up | 忠臣与奥伯伦开局不获得信息 | seen = ∅ | 随机实例 |\\n| K-SELF | — | 看到的人不含自己 | p∉seen | 随机实例 |\\n| T-LEADER | Team Building | 每次提案后队长顺时针传给下一人 | leader_{k+1} = leader_k + 1 (mod n) | T16 |\\n| T-SIZE | Team Building | 各轮出队人数见表 | \\\\|team\\\\| = SIZE[n][q] | T04–T05 |\\n| T-DISTINCT | Team Building | 每人至多一枚队伍标记 | 队员互不相同 | 随机实例 |\\n| T-VOTE | Team Building | 过半赞成才通过，平票视为否决 | approved ⇔ 2·#赞成 > n | T12–T14 |\\n| T-HAMMER | Team Building | 同一轮连续 5 次否决，坏人立即获胜 | 每轮 ≤5 次提案；第 5 次否决后游戏结束且坏人胜 | T15 |\\n| T-LAST | Team Building | 通过的提案即出队队伍，通过后不再提案 | 最后一次提案通过且其队伍 = 任务队伍 | 篡改实例 |\\n| Q-SIZE | Quest Phase | 执行任务的人数与表一致 | \\\\|team_q\\\\| = SIZE[n][q] | T06 |\\n| Q-GOOD | Quest Phase | 好人必须出成功；坏人可出成功或失败 | 对队员 i：¬evil_i → ¬fail_card_{q,i} | T17–T25, T35 |\\n| Q-COUNT | Quest Phase | 洗混后翻开任务牌，可见失败票张数 | Σᵢ fail_card_{q,i} = 记录的失败票数 | T17–T25 |\\n| Q-RESULT / Q-TWOFAIL | Quest Phase | 出现失败票即失败；7 人及以上的第 4 个任务需 2 张 | failed_q ⇔ fails_q ≥ need(n,q)，need = 2 当 q=4 且 n≥7，否则 1 | T07–T11 |\\n| R-ORDER | Quest Phase | 任务按 1→5 顺序进行（未启用 Targeting） | 第 k 条任务记录的编号 = k | 篡改实例 |\\n| G-END | Game End | 三胜或三负时游戏立即结束 | 达到 3 次成功或失败后不再有任务 | T39 |\\n| G-OUTCOME | Game End | 三个任务失败则坏人胜 | 3 次失败 ⇒ 结果为坏人胜 | 随机实例 |\\n| G-ASSASSIN | Assassinate Merlin | 三个任务成功后才进入刺杀 | 有刺杀 ⇒ 成功数 ≥ 3 | T38 |\\n| G-MERLIN-HIT | Assassinate Merlin | 刺中梅林则坏人胜，否则好人胜 | (role_a = 梅林) ⇔ 坏人胜 | T36, T37 |\\n| L-WHEN | Lady of the Lake | 第 2、3、4 个任务结束后各使用一次 | 使用轮次 ∈ {2,3,4}、递增、对应任务已结算 | — |\\n| L-FIRST | Lady of the Lake | 开局时湖中仙女在首位队长右手边的玩家手中 | 首位持有者 = 首位队长 − 1 (mod n) | T41 |\\n| L-CHAIN | Lady of the Lake | 被查验者获得湖中仙女 | 第 k 位持有者 = 第 k−1 次的被查验者 | T41 |\\n| L-PREV | Lady of the Lake | 用过湖中仙女的人不能被查验 | 被查验者 ∉ 历任持有者 | T41 |\\n| K-LADY | Lady of the Lake | 持有者得知被查验者的阵营（仅自己知道） | 持有者视角：evil_t = 查验结果 | T40 |\\n\\n规则实例以 `编号@位置` 的形式参与求解，例如 `Q-GOOD@q3:5` 表示“第 3 个任务中 6 号座位（0 起计为 5）若是好人则出成功”。不可满足时，`smt.py` 用基于删除的方法把冲突约束集最小化，输出的解释由条款实例和记录事实组成。\\n\\n## 2. 解释选择（编码变体）\\n\\n| 编号 | 问题 | 默认 | 另一种读法 | 影响（测试） |\\n|---|---|---|---|---|\\n| V-FAILCOUNT | 失败票张数是否公开 | 公开：规则书写的是洗混后翻开任务牌；avalonlogs 记录了 `numFails` | 只公开成败：有一份规则 wiki 写的是只公布任务结果 | 只公开成败时，“2 张失败 ⇒ 两人都是坏人”变为无法确定（T17 对比 T42） |\\n| V-OBERON | 梅林能否看到奥伯伦 | 能：规则书中坏人举拇指的步骤并未把奥伯伦排除 | 不能：部分规则整理认为奥伯伦对所有人隐藏 | 梅林看到 3 人在默认读法下一致，在另一读法下矛盾（T43 对比 T44） |\\n| L-FIRST | “队长右手边”指哪一位 | 队长按顺时针传递，右手边取上一座位 | 下一座位 | 只影响湖中仙女的合法性检查 |\\n| T-LEADER | 跨轮次时队长是否继续轮转 | 继续顺时针，与轮次无关 | — | 在真实日志上统计验证 |\\n\\n## 3. 未形式化的内容及原因\\n\\n| 内容 | 原因 |\\n|---|---|\\n| 讨论、声明、说谎 | 规则不约束玩家说什么，声明不是事实 |\\n| 投票与身份的关系 | 规则不限制任何人怎么投票，所以投票不携带逻辑信息（`selfcheck.py` 第 5 项验证：去掉所有投票后身份判定不变） |\\n| 坏人出成功票的策略 | 规则允许坏人出成功，因此成功的任务在逻辑上不排除坏人 |\\n| 刺客身份与刺杀前的讨论 | 不影响任何可验证结论；刺杀结果作为 G-MERLIN-HIT 建模 |\\n| Targeting 可选规则 | 真实日志所在平台未使用；R-ORDER 假设任务按顺序进行 |\\n| The Resistance 的 Plot 牌与忠诚牌 | 属于另一款游戏的扩展 |\\n| “不得出示角色牌” | 行为规范，不影响记录的一致性 |\\n| 5 人局加入派西维尔时应同时加入莫德雷德或莫甘娜 | 规则书写的是建议，作为配置提示，不作为约束 |\\n"}')
for name, content in ASSETS.items():
    p = ROOT / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')
sys.path.insert(0, str(ROOT / 'avalon'))
os.chdir(ROOT)
print('assets:', len(ASSETS))

## 1–3 数据：克隆、审计、冻结评测集

In [ ]:
import subprocess
DATA = ROOT / 'data' / 'avalonlogs'
if not DATA.exists():
    subprocess.check_call(['git', 'clone', '--quiet', 'https://github.com/WhoaWhoa/avalonlogs', str(DATA)])
COMMIT = subprocess.check_output(['git', '-C', str(DATA), 'rev-parse', 'HEAD'], text=True).strip()
print('avalonlogs commit', COMMIT)
subprocess.check_call([sys.executable, 'avalon/logs.py', '--data', str(DATA), '--inspect', '--audit',
                       '--out', 'results/audit'])
audit = json.loads((ROOT / 'results/audit/audit_summary.json').read_text())
audit['avalonlogs_commit'] = COMMIT
(ROOT / 'results/audit/audit_summary.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2))

In [ ]:
TASKS = ROOT / 'data/llm/tasks.jsonl'
if not TASKS.exists():
    subprocess.check_call([sys.executable, 'avalon/llm_tasks.py', '--parsed', 'results/audit/parsed_games.jsonl',
                           '--out', 'data/llm', '--backend', 'z3', '--seed', '137'])
manifest = json.loads((ROOT / 'data/llm/manifest.json').read_text())
assert manifest['tasks_sha256'] == hashlib.sha256(TASKS.read_bytes()).hexdigest()
manifest['avalonlogs_commit'] = COMMIT
(ROOT / 'data/llm/manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
items = [json.loads(x) for x in TASKS.read_text(encoding='utf-8').splitlines() if x.strip()]
print(json.dumps(manifest, ensure_ascii=False, indent=1))

## 4 Z3 与独立穷举交叉验证（不调用模型）

In [ ]:
r = subprocess.run([sys.executable, 'avalon/selfcheck.py', '--quick', '--out', 'results/selfcheck_colab'],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert 'ALL CHECKS PASSED' in r.stdout, 'Z3 与穷举不一致：停止，不要运行模型'
import oracle as O
from smt import SMT
smt = SMT()
small = [it for it in items if it['inst']['rec']['n'] <= 8]
bad = [it['id'] for it in small if O.Oracle().classify(it['inst'], it['claim'])['label'] != it['gold']]
print('gold re-checked by oracle on', len(small), 'items; mismatches:', bad)
assert not bad

## 5 加载固定 revision 的 Qwen3-8B

In [ ]:
from llm_run import *
runner = Runner('results/llm')
print(json.dumps(runner.meta, indent=1))

### 5a 直接判断、事实提取、湖中仙女查验（关闭 thinking，greedy）

In [ ]:
recs = {}
init = runner.run(initial_jobs(items), 'initial')
for t in ('direct', 'extract', 'reveal'):
    recs[t] = [r for r in init if r['task'] == t]
print({k: len(v) for k, v in recs.items()})

### 5b 开启 thinking（按 Qwen 推荐的采样参数）

In [ ]:
recs['think'] = runner.run(think_jobs(items, budget=3072), 'think')

### 5c 求解器反馈后修正（仅对直接判断答错的题）

In [ ]:
for mode in ('flag', 'evidence'):
    recs['revise_' + mode] = runner.run(revise_jobs(items, recs['direct'], smt, mode), 'revise_' + mode)

### 5d 评分

In [ ]:
metrics = score(items, recs, smt)
metrics['manifest'] = manifest
(ROOT / 'results/llm/metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2))
print(json.dumps({k: {kk: v[kk] for kk in ('accuracy', 'macro_f1', 'premature_certainty', 'false_uncertainty',
                                           'missed_inconsistency', 'parse_failures') if kk in v}
                  for k, v in metrics.items() if isinstance(v, dict) and 'accuracy' in v}, ensure_ascii=False, indent=1))
print(json.dumps({k: metrics.get(k) for k in ('reveal', 'verification_pipeline', 'paired_bootstrap')}, indent=1))

## 6 真实对局分析（仅 CPU，约 20–40 分钟；可以在下载第 5 步结果之后再运行）

In [ ]:
r = subprocess.run([sys.executable, 'avalon/analyze.py', '--parsed', 'results/audit/parsed_games.jsonl',
                    '--out', 'results/analysis', '--games', '600', '--completion-games', '150', '--backend', 'z3'],
                   capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])

## 7 打包下载，然后断开并删除运行时

In [ ]:
import shutil
from google.colab import files
(ROOT / 'results/COMPLETED.json').write_text(json.dumps({
    'avalonlogs_commit': COMMIT, 'tasks_sha256': manifest['tasks_sha256'],
    'records': {k: len(v) for k, v in recs.items()},
    'analysis_done': (ROOT / 'results/analysis/meta.json').exists()}, indent=2))
for big in ['results/audit/parsed_games.jsonl']:
    p = ROOT / big
    if p.exists():
        shutil.make_archive(str(ROOT / 'results/audit/parsed_games'), 'zip', p.parent, p.name)
archive = shutil.make_archive('/content/avalon_results', 'zip', ROOT, 'results')
shutil.make_archive('/content/avalon_data_llm', 'zip', ROOT, 'data/llm')
print(archive)
files.download(archive)
files.download('/content/avalon_data_llm.zip')